# DocuMind: RAG document assistant (Google Colab)

Run all cells from top to bottom (*Runtime > Run all*). A GPU is optional: *Runtime > Change runtime type > T4 GPU* makes answers faster, otherwise the CPU is used. No API keys are needed.

**What happens:** install packages, write the project files, create sample documents, run the tests, load the models (about 1 GB download the first time), and launch the Gradio app with a public link.

## 1. Install dependencies
PyTorch is already installed on Colab, so it is not reinstalled.

In [ ]:
!pip install -q "gradio>=6.0,<7" faiss-cpu sentence-transformers transformers sentencepiece pypdf python-docx pillow pytesseract pytest

## 2. Optional: OCR for scanned PDFs
Set `INSTALL_OCR = False` to skip (about 30 seconds).

In [ ]:
INSTALL_OCR = True
if INSTALL_OCR:
    !apt-get -qq install -y tesseract-ocr > /dev/null 2>&1
    !which tesseract

## 3. Create the project folder

In [ ]:
import os, sys
BASE = '/content/documind' if os.path.isdir('/content') else os.path.abspath('documind')
for d in ['rag', 'ui', 'data/uploads', 'data/vector_db', 'sample_docs', 'tests', 'scripts', 'docs']:
    os.makedirs(os.path.join(BASE, d), exist_ok=True)
os.chdir(BASE)
sys.path.insert(0, BASE)
print('Working in', os.getcwd())

## 4. Source code
Each cell below writes one file of the project (`%%writefile`).

In [ ]:
%%writefile config.py
"""Central configuration for DocuMind: paths, limits, model choices and RAG defaults."""
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

BASE_DIR = Path(os.environ.get("DOCUMIND_HOME") or Path(__file__).resolve().parent)
DATA_DIR = BASE_DIR / "data"
UPLOAD_DIR = DATA_DIR / "uploads"
VECTOR_DIR = DATA_DIR / "vector_db"
SAMPLE_DIR = BASE_DIR / "sample_docs"

ALLOWED_EXTENSIONS = (".pdf", ".docx", ".txt", ".md", ".markdown")
MAX_FILE_MB = 15
MAX_FILES_PER_UPLOAD = 25
MIN_QUESTION_CHARS = 3
MAX_QUESTION_CHARS = 1000

DEFAULT_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_MODELS = {
    "sentence-transformers/all-MiniLM-L6-v2": "MiniLM-L6 (22M params, fast, default)",
    "sentence-transformers/all-MiniLM-L12-v2": "MiniLM-L12 (33M params, slightly more accurate)",
    "BAAI/bge-small-en-v1.5": "BGE-small (33M params, strong retrieval quality)",
}

DEFAULT_GENERATION_MODEL = "google/flan-t5-base"
GENERATION_MODELS = {
    "google/flan-t5-small": "FLAN-T5 Small (77M) - fastest, weakest answers",
    "google/flan-t5-base": "FLAN-T5 Base (248M) - recommended for free Colab",
    "google/flan-t5-large": "FLAN-T5 Large (783M) - best quality, needs a GPU runtime",
}

# Max prompt size (in tokens) sent to the generator. FLAN-T5 was trained on 512 tokens
# but tolerates somewhat longer inputs; 768 keeps quality and memory in balance.
MAX_INPUT_TOKENS = 768

NOT_FOUND_MESSAGE = "I couldn't find this information in the uploaded documents."


@dataclass
class RAGSettings:
    """User-tunable settings. Passed explicitly to every pipeline call (no hidden state)."""

    chunk_size: int = 600          # characters per chunk
    chunk_overlap: int = 100       # characters shared between neighbouring chunks
    top_k: int = 4                 # chunks passed to the LLM
    similarity_threshold: float = 0.25  # minimum cosine similarity to keep a chunk
    keyword_rerank: bool = True    # blend keyword coverage into the ranking
    detailed_answers: bool = False  # longer answers
    use_ocr: bool = False          # OCR scanned PDFs (needs Tesseract)
    doc_ids: tuple = ()            # restrict retrieval to these documents (empty = all)

    def validate(self) -> "RAGSettings":
        if not 100 <= int(self.chunk_size) <= 4000:
            raise ValueError("Chunk size must be between 100 and 4000 characters.")
        if not 0 <= int(self.chunk_overlap) < int(self.chunk_size):
            raise ValueError("Chunk overlap must be at least 0 and smaller than the chunk size.")
        self.chunk_size = int(self.chunk_size)
        self.chunk_overlap = int(self.chunk_overlap)
        self.top_k = max(1, min(20, int(self.top_k)))
        self.similarity_threshold = max(0.0, min(1.0, float(self.similarity_threshold)))
        self.doc_ids = tuple(self.doc_ids or ())
        return self


def get_device(prefer: str | None = None) -> str:
    """Return 'cuda' when a GPU is available, otherwise 'cpu' (or the explicit choice)."""
    if prefer and prefer != "auto":
        return prefer
    try:
        import torch

        return "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:  # torch missing or broken: CPU-only paths still work
        return "cpu"


def ensure_dirs() -> None:
    for folder in (UPLOAD_DIR, VECTOR_DIR):
        folder.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile rag/__init__.py
"""DocuMind RAG package: loaders -> chunking -> embeddings -> vector store -> retrieval -> generator."""
from rag.pipeline import Answer, DocuMindPipeline, IngestResult, Source  # noqa: F401

In [ ]:
%%writefile rag/textutils.py
"""Small shared text helpers (tokenising, stop-words, light stemming)."""
from __future__ import annotations

import re

STOPWORDS = frozenset(
    """a about above after again all also am an and any are as at be because been before being below between
    both but by can could did do does doing down during each few for from further had has have having he her
    here hers him his how i if in into is it its itself just me more most my no nor not of off on once only or
    other our ours out over own same she should so some such than that the their theirs them then there these
    they this those through to too under until up very was we were what when where which while who whom why
    will with would you your yours tell me explain describe please much many""".split()
)

_WORD = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
_SUFFIXES = ("ingly", "edly", "ing", "ed", "s", "ly")


def stem(word: str) -> str:
    for suffix in _SUFFIXES:
        if len(word) > len(suffix) + 2 and word.endswith(suffix):
            return word[: -len(suffix)]
    return word


def tokenize(text: str) -> list[str]:
    return _WORD.findall((text or "").lower())


def content_tokens(text: str) -> list[str]:
    """Lower-cased, stop-word-free, lightly stemmed tokens."""
    return [stem(t) for t in tokenize(text) if t not in STOPWORDS and len(t) > 1]

In [ ]:
%%writefile rag/loaders.py
"""Document validation, text extraction (PDF / DOCX / TXT / Markdown) and cleaning."""
from __future__ import annotations

import hashlib
import io
import re
import shutil
import unicodedata
import zipfile
from dataclasses import dataclass, field
from pathlib import Path

from config import ALLOWED_EXTENSIONS, MAX_FILE_MB


class DocumentError(Exception):
    """A file cannot be used (unsupported, empty, corrupted, too large, ...)."""


@dataclass
class Page:
    number: int | None  # None when the format has no page concept (DOCX, TXT, MD)
    text: str


@dataclass
class LoadedDocument:
    filename: str
    file_type: str
    pages: list[Page]
    size_bytes: int
    sha256: str
    total_pages: int = 0
    warnings: list[str] = field(default_factory=list)

    @property
    def char_count(self) -> int:
        return sum(len(p.text) for p in self.pages)


# --------------------------------------------------------------------------- cleaning
_CONTROL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_SOFT_WRAP = re.compile(r"(?<!\n)\n(?!\n)(?!\s*(?:[-*\u2022#>]|\d+[.)]\s))")


def clean_text(text: str) -> str:
    """Normalise whitespace and PDF artefacts while keeping paragraph breaks."""
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\u00ad", "")
    text = _CONTROL.sub(" ", text)
    text = re.sub(r"([a-z])-\n([a-z])", r"\1\2", text)      # re-join hyphenated line breaks
    text = re.sub(r"[ \t\u00a0\u200b]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = _SOFT_WRAP.sub(" ", text)                         # join soft-wrapped lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _strip_markdown(text: str) -> str:
    text = re.sub(r"^```.*$", "", text, flags=re.M)
    text = re.sub(r"^[ ]{0,3}#{1,6}[ \t]*(.+?)[ \t]*#*[ \t]*$", r"\1\n", text, flags=re.M)
    text = re.sub(r"!\[([^\]]*)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"(\*\*|__)(.+?)\1", r"\2", text)
    text = re.sub(r"(?<![\w*])\*(?!\s)(.+?)(?<!\s)\*(?![\w*])", r"\1", text)
    text = re.sub(r"`([^`]+)`", r"\1", text)
    text = re.sub(r"^[ \t]*>[ \t]?", "", text, flags=re.M)
    text = re.sub(r"^[ \t]*([-*_][ \t]*){3,}$", "", text, flags=re.M)
    text = re.sub(r"<[^>]+>", " ", text)
    return text


# ------------------------------------------------------------------------- validation
def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def validate_file(path: str | Path, max_mb: float = MAX_FILE_MB) -> str:
    """Raise DocumentError for unusable files; return the lower-case extension otherwise."""
    p = Path(path)
    if not p.is_file():
        raise DocumentError(f"File not found: {p.name}")
    ext = p.suffix.lower()
    if ext not in ALLOWED_EXTENSIONS:
        allowed = ", ".join(ALLOWED_EXTENSIONS)
        raise DocumentError(f"Unsupported file type '{ext or 'none'}'. Allowed types: {allowed}.")
    size = p.stat().st_size
    if size == 0:
        raise DocumentError("The file is empty (0 bytes).")
    if size > max_mb * 1024 * 1024:
        raise DocumentError(f"The file is {size / 1048576:.1f} MB; the limit is {max_mb:g} MB.")
    with open(p, "rb") as fh:
        head = fh.read(1024)
    if ext == ".pdf" and b"%PDF" not in head:
        raise DocumentError("Not a valid PDF (missing PDF header).")
    if ext == ".docx" and not zipfile.is_zipfile(p):
        raise DocumentError("Not a valid DOCX file (the Word container is damaged).")
    return ext


# --------------------------------------------------------------------------------- OCR
def ocr_available() -> bool:
    """True when pytesseract, Pillow and the tesseract binary are all present."""
    try:
        import PIL  # noqa: F401
        import pytesseract  # noqa: F401
    except Exception:
        return False
    return shutil.which("tesseract") is not None


def _ocr_pdf_page(page) -> str:
    import pytesseract
    from PIL import Image

    texts = []
    for image in page.images:  # embedded scans; no poppler needed
        try:
            texts.append(pytesseract.image_to_string(Image.open(io.BytesIO(image.data))))
        except Exception:
            continue
    return "\n".join(texts)


# -------------------------------------------------------------------------- extractors
def _load_pdf(path: Path, use_ocr: bool, warnings: list[str]) -> tuple[list[Page], int]:
    from pypdf import PdfReader

    try:
        reader = PdfReader(str(path))
        if reader.is_encrypted:
            try:
                ok = reader.decrypt("")
            except Exception:
                ok = 0
            if not ok:
                raise DocumentError("The PDF is password-protected.")
        total = len(reader.pages)
    except DocumentError:
        raise
    except Exception as exc:
        raise DocumentError(f"Corrupted or unreadable PDF ({type(exc).__name__}).") from exc

    ocr_ok = use_ocr and ocr_available()
    if use_ocr and not ocr_ok:
        warnings.append("OCR was requested but Tesseract is not installed; scanned pages are skipped.")
    pages: list[Page] = []
    failed = ocr_pages = 0
    for number, page in enumerate(reader.pages, start=1):
        try:
            text = clean_text(page.extract_text() or "")
        except Exception:
            text, failed = "", failed + 1
        if not text and ocr_ok:
            try:
                text = clean_text(_ocr_pdf_page(page))
                ocr_pages += bool(text)
            except Exception:
                text = ""
        if text:
            pages.append(Page(number, text))
    if failed:
        warnings.append(f"{failed} page(s) could not be read and were skipped.")
    if ocr_pages:
        warnings.append(f"OCR was used for {ocr_pages} page(s).")
    if not pages:
        hint = "" if ocr_ok else " If this is a scanned PDF, enable OCR in Settings."
        raise DocumentError("No extractable text found in the PDF." + hint)
    if len(pages) < total:
        warnings.append(f"{total - len(pages)} page(s) had no text.")
    return pages, total


def _load_docx(path: Path) -> list[Page]:
    import docx
    from docx.table import Table
    from docx.text.paragraph import Paragraph

    try:
        document = docx.Document(str(path))
    except Exception as exc:
        raise DocumentError(f"Corrupted or unreadable DOCX ({type(exc).__name__}).") from exc

    blocks: list[str] = []
    for child in document.element.body.iterchildren():
        tag = child.tag.rsplit("}", 1)[-1]
        if tag == "p":
            para = Paragraph(child, document)
            text = para.text.strip()
            if text:
                style = (para.style.name or "") if para.style is not None else ""
                blocks.append(text + ("\n" if style.lower().startswith(("heading", "title")) else ""))
        elif tag == "tbl":
            rows = [[c.text.strip() for c in row.cells] for row in Table(child, document).rows]
            rows = [r for r in rows if any(r)]
            if len(rows) >= 2:  # header + data rows -> one self-contained sentence per row
                header = rows[0]
                for row in rows[1:]:
                    pairs = [f"{h or 'Column'}: {v}" for h, v in zip(header, row) if v]
                    blocks.append("; ".join(pairs) + ".")
            elif rows:
                blocks.append(" | ".join(rows[0]))
    text = clean_text("\n\n".join(blocks))
    if not text:
        raise DocumentError("The DOCX file contains no text.")
    return [Page(None, text)]


def _decode_text(data: bytes) -> str:
    if data.startswith((b"\xff\xfe", b"\xfe\xff")):
        return data.decode("utf-16", errors="replace")
    if b"\x00" in data[:4096]:
        raise DocumentError("The file looks binary, not plain text.")
    try:
        return data.decode("utf-8-sig")
    except UnicodeDecodeError:
        return data.decode("cp1252", errors="replace")


def _load_text(path: Path, markdown: bool) -> list[Page]:
    try:
        text = _decode_text(path.read_bytes())
    except DocumentError:
        raise
    except OSError as exc:
        raise DocumentError(f"The file could not be read ({exc.strerror or type(exc).__name__}).") from exc
    if markdown:
        text = _strip_markdown(text)
    text = clean_text(text)
    if not text:
        raise DocumentError("The file contains no readable text.")
    return [Page(None, text)]


def load_document(
    path: str | Path,
    display_name: str | None = None,
    use_ocr: bool = False,
    max_mb: float = MAX_FILE_MB,
) -> LoadedDocument:
    """Validate and extract a document. Raises DocumentError with a user-friendly message."""
    p = Path(path)
    ext = validate_file(p, max_mb)
    warnings: list[str] = []
    try:
        total = 0
        if ext == ".pdf":
            pages, total = _load_pdf(p, use_ocr, warnings)
        elif ext == ".docx":
            pages = _load_docx(p)
        else:
            pages = _load_text(p, markdown=ext in (".md", ".markdown"))
    except DocumentError:
        raise
    except Exception as exc:  # last line of defence: never crash the batch
        raise DocumentError(f"Could not read the file ({type(exc).__name__}: {exc}).") from exc
    return LoadedDocument(
        filename=display_name or p.name,
        file_type=ext.lstrip("."),
        pages=pages,
        size_bytes=p.stat().st_size,
        sha256=sha256_file(p),
        total_pages=total,
        warnings=warnings,
    )

In [ ]:
%%writefile rag/chunking.py
"""Sentence-aware overlapping chunking that preserves page numbers."""
from __future__ import annotations

import re
from dataclasses import dataclass

from rag.loaders import Page

_SENTENCE_END = re.compile(r"(?<=[.!?])[\"')\]]*\s+(?=[A-Z0-9\"'(\[])")
MIN_CHUNK_ALNUM = 15


@dataclass
class Chunk:
    text: str
    page: int | None
    chunk_index: int  # position within the document (0-based)


def validate_params(chunk_size: int, overlap: int) -> None:
    if chunk_size < 100:
        raise ValueError("Chunk size must be at least 100 characters.")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("Chunk overlap must be >= 0 and smaller than the chunk size.")


def split_sentences(text: str) -> list[str]:
    units: list[str] = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = paragraph.strip()
        if paragraph:
            units.extend(s.strip() for s in _SENTENCE_END.split(paragraph) if s and s.strip())
    return units


def _hard_split(sentence: str, size: int) -> list[str]:
    """Split an over-long sentence on whitespace so no piece exceeds `size`."""
    pieces, current = [], ""
    for word in sentence.split():
        while len(word) > size:  # a single unbroken token longer than the limit
            if current:
                pieces.append(current)
                current = ""
            pieces.append(word[:size])
            word = word[size:]
        if current and len(current) + 1 + len(word) > size:
            pieces.append(current)
            current = word
        else:
            current = f"{current} {word}".strip()
    if current:
        pieces.append(current)
    return pieces


def chunk_text(text: str, chunk_size: int = 600, overlap: int = 100) -> list[str]:
    """Pack whole sentences into chunks of <= chunk_size characters.

    Each new chunk begins with the trailing sentences of the previous one (up to `overlap`
    characters) so that facts spanning a boundary stay retrievable.
    """
    validate_params(chunk_size, overlap)
    units: list[str] = []
    for sentence in split_sentences(text):
        units.extend(_hard_split(sentence, chunk_size) if len(sentence) > chunk_size else [sentence])

    chunks: list[str] = []
    current: list[str] = []
    fresh = 0  # sentences in `current` that were not part of the previous chunk
    for unit in units:
        if current and len(" ".join(current)) + 1 + len(unit) > chunk_size:
            chunks.append(" ".join(current))
            tail: list[str] = []
            for sentence in reversed(current):
                if len(" ".join([sentence, *tail])) > overlap:
                    break
                tail.insert(0, sentence)
            current, fresh = tail, 0
            while current and len(" ".join(current)) + 1 + len(unit) > chunk_size:
                current.pop(0)
        current.append(unit)
        fresh += 1
    if current and fresh:
        chunks.append(" ".join(current))
    return chunks


def chunk_pages(pages: list[Page], chunk_size: int = 600, overlap: int = 100) -> list[Chunk]:
    """Chunk every page separately (so a chunk never spans pages) and number the chunks."""
    validate_params(chunk_size, overlap)
    result: list[Chunk] = []
    for page in pages:
        for text in chunk_text(page.text, chunk_size, overlap):
            if sum(ch.isalnum() for ch in text) < MIN_CHUNK_ALNUM:
                continue  # skip page numbers, stray symbols, etc.
            result.append(Chunk(text=text, page=page.number, chunk_index=len(result)))
    return result

In [ ]:
%%writefile rag/embeddings.py
"""Embedding generation: Sentence-Transformers (default) with an offline hashing fallback.

All embedders L2-normalise their output (so inner product == cosine similarity), process texts in
batches and cache vectors by text hash, so identical text is never embedded twice.
"""
from __future__ import annotations

import hashlib
import threading
from collections import OrderedDict
from typing import Callable

import numpy as np

from config import DEFAULT_EMBEDDING_MODEL, get_device
from rag.textutils import STOPWORDS, stem, tokenize

ProgressFn = Callable[[int, int], None]


def _normalize(vectors: np.ndarray) -> np.ndarray:
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return vectors / norms


def _is_oom(exc: BaseException) -> bool:
    return "out of memory" in str(exc).lower()


class BaseEmbedder:
    name = "base"
    is_fallback = False

    def __init__(self, batch_size: int = 32, cache_size: int = 50_000):
        self.batch_size = batch_size
        self.cache_size = cache_size
        self.generated = 0     # vectors computed by the model
        self.cache_hits = 0    # vectors served from the cache
        self._cache: "OrderedDict[str, np.ndarray]" = OrderedDict()
        self._lock = threading.RLock()

    @property
    def dim(self) -> int:  # pragma: no cover - abstract
        raise NotImplementedError

    def _embed(self, texts: list[str]) -> np.ndarray:  # pragma: no cover - abstract
        raise NotImplementedError

    def encode(self, texts: list[str], batch_size: int | None = None, progress: ProgressFn | None = None) -> np.ndarray:
        """Return a float32 array of shape (len(texts), dim) with unit-length rows."""
        texts = [t if isinstance(t, str) else str(t) for t in texts]
        if not texts:
            return np.zeros((0, self.dim), dtype=np.float32)
        keys = [hashlib.sha1(t.encode("utf-8", "ignore")).hexdigest() for t in texts]
        with self._lock:
            missing: dict[str, str] = {}
            for key, text in zip(keys, texts):
                if key in self._cache:
                    self.cache_hits += 1
                    self._cache.move_to_end(key)
                elif key not in missing:
                    missing[key] = text
            items = list(missing.items())
            size = max(1, batch_size or self.batch_size)
            for start in range(0, len(items), size):
                batch = items[start : start + size]
                vectors = _normalize(self._embed([text for _, text in batch]))
                for (key, _), vector in zip(batch, vectors):
                    self._cache[key] = vector
                self.generated += len(batch)
                if progress:
                    progress(min(start + size, len(items)), len(items))
            result = np.vstack([self._cache[k] for k in keys]).astype(np.float32)
            while len(self._cache) > self.cache_size:
                self._cache.popitem(last=False)
            return result

    def info(self) -> dict:
        return {"name": self.name, "fallback": self.is_fallback, "dim": self.dim,
                "generated": self.generated, "cache_hits": self.cache_hits}


class SentenceTransformerEmbedder(BaseEmbedder):
    """Wraps `sentence_transformers.SentenceTransformer` (GPU when available)."""

    def __init__(self, model_name: str = DEFAULT_EMBEDDING_MODEL, device: str | None = None, batch_size: int = 32):
        super().__init__(batch_size=batch_size)
        self.name = model_name
        self.device = get_device(device)
        self._model = None
        self._dim: int | None = None

    def load(self) -> "SentenceTransformerEmbedder":
        with self._lock:
            if self._model is None:
                from sentence_transformers import SentenceTransformer

                self._model = SentenceTransformer(self.name, device=self.device)
                self._dim = int(self._model.encode(["dimension probe"], convert_to_numpy=True).shape[1])
        return self

    @property
    def dim(self) -> int:
        self.load()
        return int(self._dim)

    def _embed(self, texts: list[str]) -> np.ndarray:
        self.load()
        try:
            return self._model.encode(texts, batch_size=self.batch_size, convert_to_numpy=True,
                                      normalize_embeddings=True, show_progress_bar=False)
        except RuntimeError as exc:
            if not _is_oom(exc):
                raise
        # GPU out of memory: free the cache, split the batch, and finally fall back to CPU.
        import torch

        torch.cuda.empty_cache()
        if len(texts) > 1:
            half = len(texts) // 2
            return np.vstack([self._embed(texts[:half]), self._embed(texts[half:])])
        self._model.to("cpu")
        self.device = "cpu"
        return self._model.encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)


class HashingEmbedder(BaseEmbedder):
    """Dependency-free lexical embedder (hashed bag of stems + bigrams).

    Used for unit tests and as an emergency fallback when the neural model cannot be loaded.
    It matches on shared vocabulary, not meaning, so retrieval quality is lower.
    """

    name = "hashing-fallback"
    is_fallback = True

    def __init__(self, dim: int = 384, batch_size: int = 64):
        super().__init__(batch_size=batch_size)
        self._dim = dim

    @property
    def dim(self) -> int:
        return self._dim

    def _slot(self, token: str) -> tuple[int, float]:
        digest = hashlib.md5(token.encode("utf-8")).digest()
        return int.from_bytes(digest[:4], "little") % self._dim, 1.0 if digest[4] & 1 else -1.0

    def _embed(self, texts: list[str]) -> np.ndarray:
        out = np.zeros((len(texts), self._dim), dtype=np.float32)
        for row, text in enumerate(texts):
            tokens = [stem(t) for t in tokenize(text) if t not in STOPWORDS]
            counts: dict[str, float] = {}
            for token in tokens:
                counts[token] = counts.get(token, 0.0) + 1.0
            for token, count in counts.items():
                idx, sign = self._slot(token)
                out[row, idx] += sign * (1.0 + np.log(count))
            for left, right in zip(tokens, tokens[1:]):
                idx, sign = self._slot(f"{left}_{right}")
                out[row, idx] += 0.5 * sign
        return out


def load_embedder(model_name: str = DEFAULT_EMBEDDING_MODEL, device: str | None = None,
                  allow_fallback: bool = True) -> tuple[BaseEmbedder, str | None]:
    """Load the neural embedder; on failure return the hashing fallback plus a warning."""
    try:
        return SentenceTransformerEmbedder(model_name, device).load(), None
    except Exception as exc:
        if not allow_fallback:
            raise
        warning = (f"Could not load embedding model '{model_name}' ({type(exc).__name__}: {exc}). "
                   "Using the lexical hashing fallback; retrieval quality is reduced.")
        return HashingEmbedder(), warning

In [ ]:
%%writefile rag/vectorstore.py
"""FAISS vector store with separate JSON metadata, document registry and atomic persistence."""
from __future__ import annotations

import json
import os
import threading
from datetime import datetime, timezone
from pathlib import Path

import faiss
import numpy as np

INDEX_FILE = "index.faiss"
META_FILE = "metadata.json"
FORMAT_VERSION = 1


class VectorStoreError(Exception):
    """Raised when the vector database cannot complete an operation."""


class VectorStore:
    """Exact cosine search (IndexFlatIP on normalised vectors) wrapped in IndexIDMap2.

    Chunk ids are stable integers, so deleting a document removes exactly its vectors.
    Metadata (filename, page, text, ...) lives in a plain dict persisted as JSON next to the index.
    """

    def __init__(self, directory: str | Path, dim: int, embedding_model: str = "", autosave: bool = True):
        self.directory = Path(directory)
        self.dim = int(dim)
        self.embedding_model = embedding_model
        self.autosave = autosave
        self.load_warning: str | None = None
        self.stale_documents: list[dict] = []  # documents of an index built with another model
        self._lock = threading.RLock()
        self._reset_state()
        self._load()

    # ------------------------------------------------------------------ state
    def _reset_state(self) -> None:
        self.index = faiss.IndexIDMap2(faiss.IndexFlatIP(self.dim))
        self.chunks: dict[int, dict] = {}
        self.documents: dict[str, dict] = {}
        self.next_id = 1

    def reset(self, dim: int | None = None, embedding_model: str | None = None) -> None:
        """Drop everything (optionally switching dimension / model) and persist the empty state."""
        with self._lock:
            if dim is not None:
                self.dim = int(dim)
            if embedding_model is not None:
                self.embedding_model = embedding_model
            self._reset_state()
            self._save_if_needed()

    @property
    def vector_count(self) -> int:
        return int(self.index.ntotal)

    def has_document(self, doc_id: str) -> bool:
        return doc_id in self.documents

    def find_by_filename(self, filename: str) -> dict | None:
        for record in self.documents.values():
            if record["filename"] == filename:
                return record
        return None

    def list_documents(self) -> list[dict]:
        return sorted((dict(r) for r in self.documents.values()), key=lambda r: r["added_at"])

    # --------------------------------------------------------------- mutation
    def add_document(self, record: dict, chunks: list[dict], vectors: np.ndarray) -> None:
        """Store a document's chunks and vectors atomically (all or nothing)."""
        with self._lock:
            doc_id = record["doc_id"]
            if doc_id in self.documents:
                raise VectorStoreError("This document is already indexed.")
            vectors = np.ascontiguousarray(vectors, dtype=np.float32)
            if vectors.ndim != 2 or vectors.shape != (len(chunks), self.dim):
                raise VectorStoreError(
                    f"Embedding shape {vectors.shape} does not match {len(chunks)} chunks x {self.dim} dims.")
            if not chunks:
                raise VectorStoreError("Cannot index a document without chunks.")
            ids = np.arange(self.next_id, self.next_id + len(chunks), dtype=np.int64)
            try:
                self.index.add_with_ids(vectors, ids)
            except Exception as exc:
                raise VectorStoreError(f"FAISS rejected the vectors: {exc}") from exc
            for chunk_id, chunk in zip(ids.tolist(), chunks):
                self.chunks[chunk_id] = {**chunk, "id": chunk_id, "doc_id": doc_id, "filename": record["filename"]}
            self.next_id += len(chunks)
            self.documents[doc_id] = {**record, "chunks": len(chunks),
                                      "added_at": record.get("added_at") or datetime.now(timezone.utc).isoformat()}
            self._save_if_needed()

    def delete_document(self, doc_id: str) -> int:
        """Remove a document, its vectors and its metadata. Returns the number of chunks removed."""
        with self._lock:
            if doc_id not in self.documents:
                return 0
            ids = [cid for cid, meta in self.chunks.items() if meta["doc_id"] == doc_id]
            if ids:
                self.index.remove_ids(np.asarray(ids, dtype=np.int64))
            for cid in ids:
                del self.chunks[cid]
            del self.documents[doc_id]
            self._save_if_needed()
            return len(ids)

    def clear(self) -> None:
        self.reset()

    # ----------------------------------------------------------------- search
    def search(self, vector: np.ndarray, k: int, doc_ids: set[str] | None = None) -> list[tuple[dict, float]]:
        """Return up to k (metadata, cosine score) pairs, best first, optionally restricted to docs."""
        with self._lock:
            if self.index.ntotal == 0 or k <= 0:
                return []
            query = np.ascontiguousarray(np.asarray(vector, dtype=np.float32).reshape(1, -1))
            if query.shape[1] != self.dim:
                raise VectorStoreError(f"Query has {query.shape[1]} dims; the index expects {self.dim}.")
            fetch = self.index.ntotal if doc_ids else min(k, self.index.ntotal)
            scores, ids = self.index.search(query, int(fetch))
            hits: list[tuple[dict, float]] = []
            for score, cid in zip(scores[0], ids[0]):
                meta = self.chunks.get(int(cid)) if cid >= 0 else None
                if meta is None or (doc_ids and meta["doc_id"] not in doc_ids):
                    continue
                hits.append((dict(meta), float(score)))
                if len(hits) >= k:
                    break
            return hits

    # ------------------------------------------------------------ persistence
    def _save_if_needed(self) -> None:
        if self.autosave:
            self.save()

    def save(self) -> None:
        """Write index + metadata atomically (temp file, then rename)."""
        with self._lock:
            try:
                self.directory.mkdir(parents=True, exist_ok=True)
                tmp_index = self.directory / (INDEX_FILE + ".tmp")
                tmp_meta = self.directory / (META_FILE + ".tmp")
                faiss.write_index(self.index, str(tmp_index))
                payload = {"version": FORMAT_VERSION, "dim": self.dim, "embedding_model": self.embedding_model,
                           "next_id": self.next_id, "documents": self.documents,
                           "chunks": {str(k): v for k, v in self.chunks.items()}}
                tmp_meta.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
                os.replace(tmp_index, self.directory / INDEX_FILE)
                os.replace(tmp_meta, self.directory / META_FILE)
            except Exception as exc:
                raise VectorStoreError(f"Could not save the vector database: {exc}") from exc

    def _load(self) -> None:
        index_path, meta_path = self.directory / INDEX_FILE, self.directory / META_FILE
        if not (index_path.exists() and meta_path.exists()):
            return
        try:
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
            if meta.get("dim") != self.dim or meta.get("embedding_model") != self.embedding_model:
                self.stale_documents = list(meta.get("documents", {}).values())
                self.load_warning = (f"The saved index was built with '{meta.get('embedding_model')}'; "
                                     f"it will be rebuilt for '{self.embedding_model}'.")
                return
            index = faiss.read_index(str(index_path))
            if index.d != self.dim:
                raise ValueError("index dimension mismatch")
            self.index = index
            self.chunks = {int(k): v for k, v in meta["chunks"].items()}
            self.documents = dict(meta["documents"])
            self.next_id = int(meta["next_id"])
            self._reconcile()
        except Exception as exc:
            for path in (index_path, meta_path):
                if path.exists():
                    path.replace(path.with_suffix(path.suffix + ".corrupt"))
            self._reset_state()
            self.load_warning = f"The saved vector database was unreadable ({type(exc).__name__}) and was reset."

    def _reconcile(self) -> None:
        """Remove orphan vectors / metadata so the index and registry always agree."""
        in_index = {int(i) for i in faiss.vector_to_array(self.index.id_map)}
        orphans = in_index - set(self.chunks)
        if orphans:
            self.index.remove_ids(np.asarray(sorted(orphans), dtype=np.int64))
        for cid in set(self.chunks) - in_index:
            del self.chunks[cid]
        counts: dict[str, int] = {}
        for meta in self.chunks.values():
            counts[meta["doc_id"]] = counts.get(meta["doc_id"], 0) + 1
        self.documents = {d: {**r, "chunks": counts[d]} for d, r in self.documents.items() if d in counts}
        if orphans or len(self.chunks) != len(in_index):
            self.load_warning = "Stale records were found in the saved index and removed."

    def stats(self) -> dict:
        size = sum(p.stat().st_size for p in self.directory.glob("*") if p.is_file()) if self.directory.exists() else 0
        return {"documents": len(self.documents), "chunks": len(self.chunks), "vectors": self.vector_count,
                "characters": sum(int(r.get("chars", 0)) for r in self.documents.values()), "disk_bytes": size}

In [ ]:
%%writefile rag/retrieval.py
"""Query-time retrieval: embed -> vector search -> similarity threshold -> keyword rerank."""
from __future__ import annotations

from dataclasses import dataclass

from rag.textutils import content_tokens

FOLLOW_UP_PRONOUNS = {"it", "its", "they", "them", "their", "this", "these", "those", "he", "she", "his", "her"}
FOLLOW_UP_STARTS = ("and ", "what about", "how about", "why", "also", "then", "what else")


@dataclass
class RetrievedChunk:
    chunk_id: int
    doc_id: str
    filename: str
    page: int | None
    chunk_index: int
    text: str
    score: float          # cosine similarity from the vector search
    keyword_score: float  # share of query terms present in the chunk
    final_score: float    # score used for ranking (after optional reranking)
    rank: int = 0


def contextualize_question(question: str, history: list[str] | None) -> str:
    """Make short follow-up questions self-contained using the previous user question."""
    question = question.strip()
    if not history:
        return question
    lowered = question.lower()
    words = set(lowered.replace("?", " ").split())
    is_follow_up = (
        bool(words & FOLLOW_UP_PRONOUNS)
        or lowered.startswith(FOLLOW_UP_STARTS)
        or len(content_tokens(question)) <= 2
    )
    if not is_follow_up:
        return question
    return f"{question} (regarding: {history[-1].strip()})"


class Retriever:
    def __init__(self, embedder, store):
        self.embedder = embedder
        self.store = store

    def retrieve(self, query: str, top_k: int = 4, threshold: float = 0.25, rerank: bool = True,
                 doc_ids: set[str] | None = None, candidate_factor: int = 3) -> list[RetrievedChunk]:
        query = (query or "").strip()
        if not query:
            raise ValueError("The query is empty.")
        if self.store.vector_count == 0:
            return []
        vector = self.embedder.encode([query])[0]
        hits = self.store.search(vector, k=max(top_k, 1) * candidate_factor, doc_ids=set(doc_ids) if doc_ids else None)
        query_terms = set(content_tokens(query))
        results: list[RetrievedChunk] = []
        for meta, score in hits:
            if score < threshold:
                continue
            overlap = len(query_terms & set(content_tokens(meta["text"]))) / len(query_terms) if query_terms else 0.0
            final = 0.8 * score + 0.2 * overlap if rerank else score
            results.append(RetrievedChunk(meta["id"], meta["doc_id"], meta["filename"], meta.get("page"),
                                          meta["chunk_index"], meta["text"], score, overlap, final))
        results.sort(key=lambda r: r.final_score, reverse=True)
        results = results[:top_k]
        for position, item in enumerate(results, start=1):
            item.rank = position
        return results

In [ ]:
%%writefile rag/generator.py
"""Answer generation from retrieved context: FLAN-T5 (local, free) with an extractive fallback."""
from __future__ import annotations

import math
import re
import threading
from dataclasses import dataclass
from typing import Callable

from config import DEFAULT_GENERATION_MODEL, MAX_INPUT_TOKENS, get_device
from rag.chunking import split_sentences
from rag.textutils import content_tokens

_NOT_FOUND_PHRASES = {
    "unanswerable", "unknown", "no answer", "none", "na", "not mentioned", "not specified", "not stated",
    "i dont know", "i do not know", "cannot be answered", "no information",
}


@dataclass
class GenerationResult:
    answer: str
    found: bool
    chunks_used: int
    truncated: bool = False
    model: str = ""


def is_not_found(text: str) -> bool:
    cleaned = re.sub(r"[^a-z ]", "", (text or "").lower().replace("'", "")).strip()
    return not cleaned or cleaned in _NOT_FOUND_PHRASES or cleaned.startswith("unanswerable")


def build_prompt(question: str, texts: list[str], count_tokens: Callable[[str], int],
                 truncate: Callable[[str, int], str], max_input_tokens: int = MAX_INPUT_TOKENS,
                 detailed: bool = False) -> tuple[str, int, bool]:
    """Build a grounded prompt that fits the token budget.

    Returns (prompt, number_of_chunks_included, context_was_truncated). Chunks are added in rank
    order, so the best evidence is always kept; the instruction and question are never cut.
    """
    instruction = ("Read the context and answer the question using only the information in the context. "
                   'If the context does not contain the answer, reply "unanswerable".')
    ask = "Answer in a few complete sentences:" if detailed else "Answer in one or two complete sentences:"
    footer = f"\n\nQuestion: {question}\n{ask}"
    budget = max_input_tokens - count_tokens(f"{instruction}\n\nContext:\n{footer}") - 4
    lines: list[str] = []
    used, truncated = 0, False
    for number, text in enumerate(texts, start=1):
        label = f"[{number}] "
        room = budget - used - count_tokens(label) - 1
        if room < 30 and lines:
            truncated = True
            break
        room = max(room, 30)
        size = count_tokens(text)
        if size > room:
            text, size, truncated = truncate(text, room), room, True
        lines.append(label + text)
        used += count_tokens(label) + size + 1
        if truncated:
            break
    if len(lines) < len(texts):
        truncated = True
    return f"{instruction}\n\nContext:\n" + "\n".join(lines) + footer, len(lines), truncated


def find_supporting_chunks(answer: str, texts: list[str], max_sources: int = 3) -> tuple[list[int], float]:
    """Pick the chunks whose wording best supports the answer.

    Returns (1-based chunk numbers, best coverage). Coverage is the share of the answer's content
    words found in the best chunk; a low value signals an ungrounded answer.
    """
    explicit = [int(n) for n in re.findall(r"\[(\d{1,2})\]", answer) if 1 <= int(n) <= len(texts)]
    answer_terms = set(content_tokens(re.sub(r"\[\d{1,2}\]", " ", answer)))
    if not answer_terms:
        return (sorted(set(explicit)) or [1]), 1.0
    scores = [len(answer_terms & set(content_tokens(t))) / len(answer_terms) for t in texts]
    best = max(scores)
    if best == 0:
        return (sorted(set(explicit)) or [1]), 0.0
    ranked = sorted(range(len(texts)), key=lambda i: scores[i], reverse=True)
    picked = [i + 1 for i in ranked if scores[i] >= max(0.3, 0.6 * best)][:max_sources]
    return sorted(set(picked) | set(explicit)), best


class ExtractiveGenerator:
    """Model-free fallback: returns the most relevant sentences from the retrieved chunks.

    Always grounded (it only copies source text), but not fluent or abstractive.
    """

    name = "extractive-fallback"
    is_fallback = True
    device = "cpu"
    loaded = True

    def load(self) -> "ExtractiveGenerator":
        return self

    def generate(self, question: str, chunks, detailed: bool = False, max_new_tokens: int | None = None) -> GenerationResult:
        terms = set(content_tokens(question))
        sentences = [(ci, si, sent, set(content_tokens(sent)))
                     for ci, chunk in enumerate(chunks) for si, sent in enumerate(split_sentences(chunk.text))]
        # Rarer query words (in the retrieved text) count more than words that appear everywhere.
        weight = {t: math.log(1 + len(sentences) / (1 + sum(t in toks for *_, toks in sentences))) for t in terms}
        total = sum(weight.values())
        scored = []
        for ci, si, sent, toks in sentences:
            matched = sum(weight[t] for t in terms & toks)
            if total and matched > 0:
                scored.append((matched / total, -ci, -si, ci, si, sent))
        if not scored:
            return GenerationResult("", False, len(chunks), model=self.name)
        scored.sort(reverse=True)
        best = scored[0][0]
        if best < 0.34:
            return GenerationResult("", False, len(chunks), model=self.name)
        limit = 3 if detailed else 2
        picked, seen = [], set()
        for item in scored:
            if item[0] >= 0.5 * best and item[5] not in seen:
                seen.add(item[5])
                picked.append(item)
            if len(picked) >= limit:
                break
        picked.sort(key=lambda it: (it[3], it[4]))
        return GenerationResult(" ".join(p[5] for p in picked), True, len(chunks), model=self.name)


class FlanT5Generator:
    """Local instruction-tuned seq2seq model (FLAN-T5). Loaded lazily; GPU used when available."""

    is_fallback = False

    def __init__(self, model_name: str = DEFAULT_GENERATION_MODEL, device: str | None = None,
                 max_input_tokens: int = MAX_INPUT_TOKENS):
        self.name = model_name
        self.device = get_device(device)
        self.max_input_tokens = max_input_tokens
        self.loaded = False
        self._tok = None
        self._model = None
        self._lock = threading.RLock()

    def load(self) -> "FlanT5Generator":
        with self._lock:
            if self.loaded:
                return self
            from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

            self._tok = AutoTokenizer.from_pretrained(self.name)
            self._model = AutoModelForSeq2SeqLM.from_pretrained(self.name)  # float32: T5 is unstable in fp16
            self._model.to(self.device).eval()
            self.loaded = True
        return self

    # token helpers -------------------------------------------------------------
    def _count(self, text: str) -> int:
        return len(self._tok(text, add_special_tokens=False).input_ids)

    def _truncate(self, text: str, tokens: int) -> str:
        return self._tok.decode(self._tok(text, add_special_tokens=False).input_ids[:tokens], skip_special_tokens=True)

    def _run(self, prompt: str, max_new_tokens: int, beams: int) -> str:
        import torch

        inputs = self._tok(prompt, return_tensors="pt", truncation=True, max_length=self.max_input_tokens)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.inference_mode():
            output = self._model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=beams,
                                          no_repeat_ngram_size=3, early_stopping=True, do_sample=False)
        return self._tok.decode(output[0], skip_special_tokens=True).strip()

    def generate(self, question: str, chunks, detailed: bool = False, max_new_tokens: int | None = None) -> GenerationResult:
        self.load()
        max_new = max_new_tokens or (256 if detailed else 128)
        texts = [c.text for c in chunks]
        prompt, used, truncated = build_prompt(question, texts, self._count, self._truncate,
                                               self.max_input_tokens, detailed)
        try:
            raw = self._run(prompt, max_new, beams=4)
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            import torch

            torch.cuda.empty_cache()
            try:  # retry with greedy decoding (much less memory)
                raw = self._run(prompt, max_new, beams=1)
            except RuntimeError as exc2:
                if "out of memory" not in str(exc2).lower():
                    raise
                torch.cuda.empty_cache()
                self._model.to("cpu")  # last resort: continue on CPU
                self.device = "cpu"
                raw = self._run(prompt, max_new, beams=1)
        if is_not_found(raw):
            return GenerationResult("", False, used, truncated, self.name)
        return GenerationResult(raw, True, used, truncated, self.name)

In [ ]:
%%writefile rag/pipeline.py
"""DocuMind pipeline: ingestion (upload -> FAISS) and question answering (question -> answer + citations)."""
from __future__ import annotations

import re
import shutil
import threading
import time
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable

from config import (DATA_DIR, DEFAULT_EMBEDDING_MODEL, DEFAULT_GENERATION_MODEL, MAX_FILES_PER_UPLOAD,
                    MAX_QUESTION_CHARS, MIN_QUESTION_CHARS, NOT_FOUND_MESSAGE, RAGSettings, get_device)
from rag.chunking import chunk_pages
from rag.embeddings import BaseEmbedder, load_embedder
from rag.generator import ExtractiveGenerator, FlanT5Generator, find_supporting_chunks
from rag.loaders import DocumentError, load_document
from rag.retrieval import Retriever, RetrievedChunk, contextualize_question
from rag.vectorstore import VectorStore, VectorStoreError

Progress = Callable[[float, str], None]


@dataclass
class IngestResult:
    filename: str
    status: str            # "indexed" | "replaced" | "duplicate" | "failed"
    message: str = ""
    doc_id: str = ""
    pages: int = 0
    chunks: int = 0
    chars: int = 0
    seconds: float = 0.0
    warnings: list[str] = field(default_factory=list)


@dataclass
class Source:
    number: int
    filename: str
    page: int | None
    chunk_index: int
    text: str
    score: float
    final_score: float
    cited: bool


@dataclass
class Answer:
    question: str
    text: str
    kind: str                       # "answer" | "not_found" | "empty" | "invalid" | "error"
    found: bool = False
    sources: list[Source] = field(default_factory=list)
    standalone_question: str = ""
    model: str = ""
    seconds: float = 0.0
    low_confidence: bool = False
    truncated: bool = False
    warnings: list[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        return asdict(self)


def safe_filename(name: str) -> str:
    base = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(name).name).strip("._") or "document"
    return base[-120:]


class DocuMindPipeline:
    """Coordinates loaders, chunking, embeddings, the vector store, retrieval and generation."""

    def __init__(self, data_dir: str | Path | None = None, embedding_model: str = DEFAULT_EMBEDDING_MODEL,
                 generation_model: str = DEFAULT_GENERATION_MODEL, device: str | None = None,
                 embedder: BaseEmbedder | None = None, generator=None):
        self.data_dir = Path(data_dir or DATA_DIR)
        self.upload_dir = self.data_dir / "uploads"
        self.vector_dir = self.data_dir / "vector_db"
        for folder in (self.upload_dir, self.vector_dir):
            folder.mkdir(parents=True, exist_ok=True)
        self.device = get_device(device)
        self.warnings: list[str] = []
        self.busy = ""  # human-readable activity ("Indexing 3 files..."); empty when idle
        self._lock = threading.RLock()

        if embedder is None:
            embedder, warning = load_embedder(embedding_model, self.device)
            if warning:
                self.warnings.append(warning)
        self.embedder = embedder
        self.generator = generator or FlanT5Generator(generation_model, self.device)
        self.store = VectorStore(self.vector_dir, self.embedder.dim, self.embedder.name)
        self.retriever = Retriever(self.embedder, self.store)
        if self.store.load_warning:
            self.warnings.append(self.store.load_warning)
        if self.store.stale_documents:
            self._reindex_records(self.store.stale_documents, RAGSettings())

    # ---------------------------------------------------------------- models
    @property
    def generator_ready(self) -> bool:
        return bool(getattr(self.generator, "loaded", True))

    def load_models(self, progress: Progress | None = None) -> list[str]:
        """Load the generator now (instead of on the first question). Returns warnings."""
        if progress:
            progress(0.1, f"Loading {self.generator.name}...")
        self.ensure_generator()
        if progress:
            progress(1.0, "Models ready")
        return list(self.warnings)

    def ensure_generator(self) -> None:
        with self._lock:
            if self.generator_ready:
                return
            try:
                self.generator.load()
            except Exception as exc:
                self.warnings.append(f"Could not load '{self.generator.name}' ({type(exc).__name__}: {exc}). "
                                     "Answers fall back to extractive mode (copied passages).")
                self.generator = ExtractiveGenerator()

    def set_generation_model(self, name: str) -> str:
        """Switch the LLM. Keeps the current one if the new model cannot be loaded."""
        with self._lock:
            if name == getattr(self.generator, "name", ""):
                return f"{name} is already active."
            candidate = FlanT5Generator(name, self.device)
            try:
                candidate.load()
            except Exception as exc:
                return f"Could not load {name}: {type(exc).__name__}: {exc}. Keeping {self.generator.name}."
            self.generator = candidate
            return f"Switched generation model to {name}."

    def set_embedding_model(self, name: str, settings: RAGSettings | None = None,
                            progress: Progress | None = None) -> str:
        """Switch the embedding model and rebuild the index for all stored documents."""
        with self._lock:
            if name == self.embedder.name:
                return f"{name} is already active."
            embedder, warning = load_embedder(name, self.device)
            if warning:
                return warning + " The previous model stays active."
            records = self.store.list_documents()
            self.embedder = embedder
            self.retriever.embedder = embedder
            self.store.reset(embedder.dim, embedder.name)
            self._reindex_records(records, settings or RAGSettings(), progress)
            return f"Switched embedding model to {name} and re-indexed {len(records)} document(s)."

    def status(self) -> dict:
        return {"embedding_model": self.embedder.name, "embedding_fallback": self.embedder.is_fallback,
                "generation_model": getattr(self.generator, "name", "?"),
                "generation_fallback": getattr(self.generator, "is_fallback", False),
                "generator_loaded": self.generator_ready, "device": self.device,
                "embeddings_generated": self.embedder.generated, "cache_hits": self.embedder.cache_hits,
                "busy": self.busy, "warnings": list(self.warnings)}

    # ------------------------------------------------------------- ingestion
    def ingest_files(self, paths: list[str | Path], settings: RAGSettings | None = None,
                     progress: Progress | None = None) -> list[IngestResult]:
        settings = (settings or RAGSettings()).validate()
        results: list[IngestResult] = []
        paths = list(paths or [])
        total = max(len(paths), 1)
        for position, path in enumerate(paths):
            name = Path(path).name
            if position >= MAX_FILES_PER_UPLOAD:
                results.append(IngestResult(name, "failed", f"Skipped: at most {MAX_FILES_PER_UPLOAD} files per upload."))
                continue
            lo, hi = position / total, (position + 1) / total

            def step(fraction: float, text: str, lo=lo, hi=hi, name=name) -> None:
                if progress:
                    progress(lo + (hi - lo) * fraction, f"{name}: {text}")

            results.append(self.ingest_file(path, settings, progress=step))
        if progress:
            progress(1.0, "Done")
        return results

    def ingest_file(self, path: str | Path, settings: RAGSettings | None = None, display_name: str | None = None,
                    progress: Progress | None = None) -> IngestResult:
        """Run one file through Extract -> Clean -> Chunk -> Metadata -> Embed -> FAISS."""
        settings = (settings or RAGSettings()).validate()
        name = display_name or Path(path).name
        started = time.time()
        step = progress or (lambda fraction, text: None)
        try:
            step(0.05, "extracting text")
            doc = load_document(path, display_name=name, use_ocr=settings.use_ocr)
        except DocumentError as exc:
            return IngestResult(name, "failed", str(exc), seconds=time.time() - started)
        doc_id = doc.sha256[:16]

        with self._lock:
            existing = self.store.documents.get(doc_id)
            if existing:
                return IngestResult(name, "duplicate", f"Already indexed as '{existing['filename']}'.", doc_id,
                                    seconds=time.time() - started)
            step(0.2, "splitting into chunks")
            chunks = chunk_pages(doc.pages, settings.chunk_size, settings.chunk_overlap)
            if not chunks:
                return IngestResult(name, "failed", "The document has too little text to index.", doc_id,
                                    seconds=time.time() - started)
            try:
                vectors = self.embedder.encode(
                    [c.text for c in chunks],
                    progress=lambda done, total: step(0.25 + 0.6 * done / max(total, 1), f"embedding {done}/{total}"))
            except Exception as exc:
                return IngestResult(name, "failed", f"Embedding failed ({type(exc).__name__}: {exc}).", doc_id,
                                    seconds=time.time() - started)

            stored_as = f"{doc_id}_{safe_filename(name)}"
            dest = self.upload_dir / stored_as
            try:
                if Path(path).resolve() != dest.resolve():
                    shutil.copy2(path, dest)
                step(0.9, "saving to the vector database")
                previous = self.store.find_by_filename(name)
                record = {"doc_id": doc_id, "filename": name, "file_type": doc.file_type,
                          "size_bytes": doc.size_bytes, "pages": doc.total_pages, "chars": doc.char_count,
                          "stored_as": stored_as, "added_at": datetime.now(timezone.utc).isoformat(),
                          "chunk_size": settings.chunk_size, "chunk_overlap": settings.chunk_overlap}
                chunk_dicts = [{"page": c.page, "chunk_index": c.chunk_index, "text": c.text} for c in chunks]
                self.store.add_document(record, chunk_dicts, vectors)
                if previous:  # same filename, new content: replace the stale version
                    self._remove_files(previous)
                    self.store.delete_document(previous["doc_id"])
            except (VectorStoreError, OSError) as exc:
                if dest.exists() and not self.store.has_document(doc_id):
                    dest.unlink(missing_ok=True)
                return IngestResult(name, "failed", f"Could not store the document: {exc}", doc_id,
                                    seconds=time.time() - started)
        status = "replaced" if previous else "indexed"
        message = "Replaced the previous version of this file." if previous else "Indexed."
        return IngestResult(name, status, message, doc_id, doc.total_pages, len(chunks), doc.char_count,
                            time.time() - started, doc.warnings)

    def _remove_files(self, record: dict) -> None:
        stored = record.get("stored_as")
        if stored:
            (self.upload_dir / stored).unlink(missing_ok=True)

    def _reindex_records(self, records: list[dict], settings: RAGSettings,
                         progress: Progress | None = None) -> list[IngestResult]:
        results = []
        total = max(len(records), 1)
        for position, record in enumerate(records):
            path = self.upload_dir / record.get("stored_as", "")
            if not path.is_file():
                results.append(IngestResult(record["filename"], "failed", "The stored original file is missing."))
                continue
            if progress:
                progress(position / total, f"Re-indexing {record['filename']}")
            results.append(self.ingest_file(path, settings, display_name=record["filename"]))
        return results

    def reindex_all(self, settings: RAGSettings | None = None, progress: Progress | None = None) -> list[IngestResult]:
        """Re-chunk and re-embed every stored document with the given settings."""
        settings = (settings or RAGSettings()).validate()
        with self._lock:
            records = self.store.list_documents()
            self.store.reset()
            return self._reindex_records(records, settings, progress)

    # ------------------------------------------------------------ management
    def list_documents(self) -> list[dict]:
        return self.store.list_documents()

    def delete_document(self, doc_id: str) -> bool:
        with self._lock:
            record = self.store.documents.get(doc_id)
            if not record:
                return False
            self.store.delete_document(doc_id)
            self._remove_files(record)
            return True

    def clear_repository(self) -> int:
        with self._lock:
            count = len(self.store.documents)
            self.store.clear()
            for path in self.upload_dir.glob("*"):
                if path.is_file() and path.name != ".gitkeep":
                    path.unlink(missing_ok=True)
            return count

    def stats(self) -> dict:
        return {**self.store.stats(), **self.status()}

    # -------------------------------------------------------------- question
    def ask(self, question: str, settings: RAGSettings | None = None, history: list[str] | None = None) -> Answer:
        """Answer a question from the indexed documents, with citations."""
        started = time.time()
        question = (question or "").strip()
        if len(question) < MIN_QUESTION_CHARS or not any(ch.isalnum() for ch in question):
            return Answer(question, "Please enter a question of at least a few words.", "invalid")
        if len(question) > MAX_QUESTION_CHARS:
            return Answer(question[:80] + "...", f"That question is too long (limit {MAX_QUESTION_CHARS} characters).", "invalid")
        if self.store.vector_count == 0:
            return Answer(question, "The repository is empty. Upload documents (or load the sample set) first.", "empty")
        settings = (settings or RAGSettings()).validate()

        standalone = contextualize_question(question, history)
        retrieved = self.retriever.retrieve(standalone, settings.top_k, settings.similarity_threshold,
                                            settings.keyword_rerank, set(settings.doc_ids) or None)
        if not retrieved:
            return Answer(question, NOT_FOUND_MESSAGE, "not_found", standalone_question=standalone,
                          seconds=time.time() - started)

        warnings: list[str] = []
        self.ensure_generator()
        try:
            result = self.generator.generate(standalone, retrieved, detailed=settings.detailed_answers)
        except Exception as exc:  # model crashed: degrade to extractive answers instead of failing
            warnings.append(f"The language model failed ({type(exc).__name__}); showing extracted passages instead.")
            result = ExtractiveGenerator().generate(standalone, retrieved, detailed=settings.detailed_answers)
        used = retrieved[: max(result.chunks_used, 1)]
        model = getattr(result, "model", "") or getattr(self.generator, "name", "")

        if not result.found:
            return Answer(question, NOT_FOUND_MESSAGE, "not_found", False, self._sources(used, set()),
                          standalone, model, time.time() - started, truncated=result.truncated, warnings=warnings)

        cited, coverage = find_supporting_chunks(result.answer, [c.text for c in used])
        text = re.sub(r"\s*\[\d{1,2}\]", "", result.answer).strip()
        low = len(text.split()) >= 4 and coverage < 0.25
        return Answer(question, text, "answer", True, self._sources(used, set(cited)), standalone, model,
                      time.time() - started, low_confidence=low, truncated=result.truncated, warnings=warnings)

    @staticmethod
    def _sources(chunks: list[RetrievedChunk], cited: set[int]) -> list[Source]:
        return [Source(i, c.filename, c.page, c.chunk_index, c.text, round(c.score, 4), round(c.final_score, 4), i in cited)
                for i, c in enumerate(chunks, start=1)]

In [ ]:
%%writefile ui/__init__.py
"""DocuMind user interface (Gradio)."""

In [ ]:
%%writefile ui/components.py
"""Shared UI pieces: theme, CSS and HTML builders (kept free of Gradio event logic)."""
from __future__ import annotations

import html
import re
from datetime import datetime

import gradio as gr

from config import RAGSettings
from rag.textutils import STOPWORDS, tokenize

esc = html.escape

SUGGESTIONS = [
    "How much is the home-office stipend?",
    "What is the difference between supervised and unsupervised learning?",
    "Where does the Calvin cycle take place?",
    "What are the main challenges of wind and solar power?",
]

FORCE_LIGHT_JS = "() => { document.documentElement.classList.remove('dark'); document.body.classList.remove('dark'); }"

CSS = """
:root{--dm-bg:#F5F6FB;--dm-card:#FFFFFF;--dm-ink:#1A1D3B;--dm-muted:#667085;--dm-line:#E3E6F2;
--dm-blue:#3F6BF4;--dm-violet:#7B5CF0;--dm-blue-soft:#EAF0FF;--dm-violet-soft:#F1ECFF;
--dm-ok:#0F7B55;--dm-ok-soft:#E6F6EF;--dm-warn:#9A5B06;--dm-warn-soft:#FFF4E0;--dm-bad:#B42318;--dm-bad-soft:#FDECEA}
.gradio-container{max-width:1360px!important;margin:0 auto!important;background:var(--dm-bg)!important;color:var(--dm-ink)}
footer{display:none!important}
.dm-layout{gap:20px!important;align-items:flex-start!important}
.dm-side{background:var(--dm-card);border:1px solid var(--dm-line);border-radius:20px;padding:18px!important;gap:12px!important}
@media (max-width:900px){.dm-layout{flex-direction:column!important}.dm-side{min-width:100%!important}}
.dm-logo{display:flex;gap:12px;align-items:center}
.dm-logo svg{flex:none}
.dm-logo b{font-size:20px;letter-spacing:-.02em;display:block}
.dm-logo span{font-size:12.5px;color:var(--dm-muted);line-height:1.35;display:block;max-width:230px}
.dm-h{font-weight:700;font-size:13.5px;margin:6px 0 0;color:var(--dm-ink)}
.dm-hero{background:var(--dm-card);border:1px solid var(--dm-line);border-radius:20px;padding:20px 24px;display:flex;
justify-content:space-between;gap:16px;align-items:flex-start;border-top:3px solid var(--dm-violet)}
.dm-hero h1{font-size:26px;letter-spacing:-.03em;margin:0 0 4px;font-weight:800}
.dm-hero p{margin:0;color:var(--dm-muted);font-size:14px;max-width:560px;line-height:1.5}
.dm-stats{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;margin-top:12px}
@media (max-width:700px){.dm-stats{grid-template-columns:repeat(2,minmax(0,1fr))}}
.dm-stat{background:var(--dm-card);border:1px solid var(--dm-line);border-radius:16px;padding:14px 16px}
.dm-stat small{display:block;color:var(--dm-muted);font-size:12px;margin-bottom:2px}
.dm-stat strong{font-size:24px;letter-spacing:-.02em;font-variant-numeric:tabular-nums}
.dm-stat em{display:block;font-style:normal;font-size:11.5px;color:var(--dm-muted);margin-top:2px;overflow-wrap:anywhere}
.dm-pill{display:inline-flex;align-items:center;gap:6px;border-radius:999px;padding:4px 11px;font-size:12.5px;font-weight:600}
.dm-pill::before{content:"";width:7px;height:7px;border-radius:50%;background:currentColor}
.dm-pill-ok{background:var(--dm-ok-soft);color:var(--dm-ok)}.dm-pill-busy{background:var(--dm-blue-soft);color:var(--dm-blue)}
.dm-pill-warn{background:var(--dm-warn-soft);color:var(--dm-warn)}
.dm-note{border-radius:12px;padding:9px 12px;font-size:13px;line-height:1.45;margin:6px 0}
.dm-note-info{background:var(--dm-blue-soft);color:#24408F}.dm-note-ok{background:var(--dm-ok-soft);color:var(--dm-ok)}
.dm-note-warn{background:var(--dm-warn-soft);color:var(--dm-warn)}.dm-note-error{background:var(--dm-bad-soft);color:var(--dm-bad)}
.dm-log{display:flex;flex-direction:column;gap:6px}
.dm-log-row{border-radius:12px;padding:8px 11px;font-size:12.5px;line-height:1.4;overflow-wrap:anywhere}
.dm-log-row b{display:block;font-size:13px}
.dm-doc{display:flex;gap:10px;align-items:center;padding:9px 0;border-bottom:1px solid var(--dm-line)}
.dm-doc:last-child{border-bottom:none}
.dm-ext{flex:none;width:38px;height:38px;border-radius:10px;display:grid;place-items:center;font-size:10.5px;font-weight:800}
.dm-ext-a{background:var(--dm-blue-soft);color:var(--dm-blue)}.dm-ext-b{background:var(--dm-violet-soft);color:var(--dm-violet)}
.dm-doc-main{min-width:0}.dm-doc-main b{display:block;font-size:13px;overflow-wrap:anywhere}
.dm-doc-main span{font-size:11.5px;color:var(--dm-muted)}
.dm-empty{color:var(--dm-muted);font-size:13px;line-height:1.5;padding:6px 0}
.dm-kv{display:grid;grid-template-columns:1fr auto;gap:5px 10px;font-size:12.5px}
.dm-kv span{color:var(--dm-muted)}.dm-kv b{font-variant-numeric:tabular-nums;text-align:right}
.dm-chatbot{border-radius:18px!important}
.dm-sugg-title{font-size:12.5px;color:var(--dm-muted);margin:4px 0}
.dm-chip-btn{text-align:left!important;white-space:normal!important;height:auto!important;font-size:12.5px!important}
.dm-answer-text{font-size:15px;line-height:1.6}
.dm-ref{display:inline-grid;place-items:center;min-width:20px;height:20px;border-radius:10px;background:var(--dm-violet);color:#fff;
font-size:11.5px;font-weight:700;margin-left:3px;vertical-align:middle;padding:0 5px}
.dm-meta{display:flex;flex-wrap:wrap;gap:4px 14px;font-size:12px;color:var(--dm-muted);margin:8px 0 2px}
.dm-sources{margin-top:10px;display:flex;flex-direction:column;gap:7px}
.dm-sources-title{font-weight:700;font-size:13px}
.dm-src{border:1px solid var(--dm-line);border-radius:14px;background:#FBFBFE;overflow:hidden}
.dm-src[open]{border-color:#C9D3FA;background:#fff}
.dm-src summary{cursor:pointer;display:flex;flex-wrap:wrap;gap:6px 10px;align-items:center;padding:9px 12px;list-style:none;font-size:13px}
.dm-src summary::-webkit-details-marker{display:none}
.dm-badge{display:inline-grid;place-items:center;width:24px;height:24px;border-radius:50%;font-size:12px;font-weight:700;
border:1.5px solid var(--dm-violet);color:var(--dm-violet);flex:none}
.dm-badge-on{background:var(--dm-violet);color:#fff}
.dm-src-name{font-weight:600;overflow-wrap:anywhere}
.dm-chip{background:var(--dm-blue-soft);color:var(--dm-blue);border-radius:8px;padding:2px 8px;font-size:11.5px;font-weight:600}
.dm-bar{display:inline-block;width:70px;height:6px;border-radius:3px;background:var(--dm-line);overflow:hidden}
.dm-bar i{display:block;height:100%;background:linear-gradient(90deg,var(--dm-blue),var(--dm-violet))}
.dm-score{font-size:12px;color:var(--dm-muted);font-variant-numeric:tabular-nums}
.dm-passage{padding:2px 14px 12px 14px;font-size:13px;line-height:1.6;color:#2D3152;border-top:1px dashed var(--dm-line)}
.dm-passage mark{background:#E6DEFF;color:inherit;border-radius:3px;padding:0 1px}
.dm-typing{display:inline-flex;gap:5px;padding:6px 2px}
.dm-typing span{width:8px;height:8px;border-radius:50%;background:var(--dm-violet);opacity:.35;animation:dmblink 1s infinite}
.dm-typing span:nth-child(2){animation-delay:.15s}.dm-typing span:nth-child(3){animation-delay:.3s}
@keyframes dmblink{0%,100%{opacity:.25}50%{opacity:1}}
@media (prefers-reduced-motion:reduce){.dm-typing span{animation:none;opacity:.7}}
.dm-flow{display:flex;flex-wrap:wrap;gap:8px;align-items:center;margin:8px 0 18px}
.dm-flow span{background:var(--dm-card);border:1px solid var(--dm-line);border-radius:10px;padding:6px 11px;font-size:13px;font-weight:600}
.dm-flow i{color:var(--dm-muted);font-style:normal}
.dm-doc-text{font-size:14px;line-height:1.65;max-width:760px}
"""


def make_theme() -> gr.themes.Base:
    return gr.themes.Base(
        primary_hue="indigo", secondary_hue="violet", neutral_hue="slate", radius_size="lg",
        font=[gr.themes.GoogleFont("Plus Jakarta Sans"), "ui-sans-serif", "system-ui", "sans-serif"],
    ).set(
        body_background_fill="#F5F6FB",
        button_primary_background_fill="linear-gradient(135deg,#3F6BF4,#7B5CF0)",
        button_primary_background_fill_hover="linear-gradient(135deg,#3860E6,#6F4FE6)",
        button_primary_text_color="white",
    )


# ----------------------------------------------------------------------------- helpers
def human_size(num: float) -> str:
    for unit in ("B", "KB", "MB", "GB"):
        if num < 1024 or unit == "GB":
            return f"{num:.0f} {unit}" if unit == "B" else f"{num:.1f} {unit}"
        num /= 1024
    return f"{num} B"


def settings_from_inputs(chunk_size=600, overlap=100, top_k=4, threshold=0.25, rerank=True,
                         detailed=False, ocr=False, doc_ids=()) -> RAGSettings:
    return RAGSettings(chunk_size=int(chunk_size), chunk_overlap=int(overlap), top_k=int(top_k),
                       similarity_threshold=float(threshold), keyword_rerank=bool(rerank),
                       detailed_answers=bool(detailed), use_ocr=bool(ocr), doc_ids=tuple(doc_ids or ())).validate()


def status_html(message: str, kind: str = "info") -> str:
    return f'<div class="dm-note dm-note-{kind}">{esc(message)}</div>'


def logo_html() -> str:
    mark = ('<svg width="44" height="44" viewBox="0 0 44 44" aria-hidden="true"><defs><linearGradient id="dmg" x1="0" y1="0" x2="1" y2="1">'
            '<stop offset="0" stop-color="#3F6BF4"/><stop offset="1" stop-color="#7B5CF0"/></linearGradient></defs>'
            '<rect width="44" height="44" rx="13" fill="url(#dmg)"/><path d="M13 11h13l6 6v16a2 2 0 0 1-2 2H13a2 2 0 0 1-2-2V13a2 2 0 0 1 2-2z" fill="#fff" fill-opacity=".95"/>'
            '<path d="M16 22h12M16 26h12M16 30h7" stroke="#5B5FE8" stroke-width="2" stroke-linecap="round"/></svg>')
    return (f'<div class="dm-logo">{mark}<div><b>DocuMind</b>'
            '<span>Ask questions across your documents. Every answer shows where it came from.</span></div></div>')


def _status_pill(stats: dict) -> str:
    if stats.get("busy"):
        return f'<span class="dm-pill dm-pill-busy">{esc(stats["busy"])}</span>'
    if stats.get("embedding_fallback") or stats.get("generation_fallback") or stats.get("warnings"):
        return '<span class="dm-pill dm-pill-warn">Ready with warnings</span>'
    return '<span class="dm-pill dm-pill-ok">Ready</span>'


def hero_html(stats: dict) -> str:
    models = f'{stats["embedding_model"].split("/")[-1]} + {stats["generation_model"].split("/")[-1]} on {stats["device"]}'
    cards = [("Documents", stats["documents"], ""), ("Chunks", stats["chunks"], ""),
             ("Vectors", stats["vectors"], "")]
    cards_html = "".join(f'<div class="dm-stat"><small>{t}</small><strong>{v:,}</strong></div>' for t, v, _ in cards)
    cards_html += f'<div class="dm-stat"><small>Status</small>{_status_pill(stats)}<em>{esc(models)}</em></div>'
    return ('<div class="dm-hero"><div><h1>Ask your documents</h1>'
            '<p>Upload files, ask in plain language, and get answers grounded in your own text, '
            'with the exact passages they came from.</p></div></div>'
            f'<div class="dm-stats">{cards_html}</div>')


def stats_html(stats: dict) -> str:
    rows = [("Documents", f'{stats["documents"]:,}'), ("Chunks", f'{stats["chunks"]:,}'),
            ("Vectors in FAISS", f'{stats["vectors"]:,}'), ("Characters indexed", f'{stats["characters"]:,}'),
            ("Index size on disk", human_size(stats["disk_bytes"])),
            ("Embeddings generated", f'{stats["embeddings_generated"]:,}'), ("Served from cache", f'{stats["cache_hits"]:,}')]
    return '<div class="dm-kv">' + "".join(f"<span>{k}</span><b>{v}</b>" for k, v in rows) + "</div>"


def library_html(docs: list[dict]) -> str:
    if not docs:
        return '<div class="dm-empty">No documents yet. Upload files above, or choose Load samples to try DocuMind right away.</div>'
    items = []
    for d in docs:
        ext = d.get("file_type", "").upper()[:4]
        tone = "a" if d.get("file_type") in ("pdf", "docx") else "b"
        pages = f'{d["pages"]} pages, ' if d.get("pages") else ""
        try:
            added = datetime.fromisoformat(d["added_at"]).strftime("%d %b %H:%M")
        except Exception:
            added = ""
        items.append(f'<div class="dm-doc"><div class="dm-ext dm-ext-{tone}">{esc(ext)}</div><div class="dm-doc-main">'
                     f'<b>{esc(d["filename"])}</b><span>{pages}{d["chunks"]} chunks, {human_size(d["size_bytes"])}'
                     f'{", added " + added if added else ""}</span></div></div>')
    return "".join(items)


def upload_log_html(results) -> str:
    if not results:
        return status_html("No files were processed.", "warn")
    palette = {"indexed": ("ok", "Indexed"), "replaced": ("ok", "Updated"), "duplicate": ("warn", "Skipped"), "failed": ("error", "Failed")}
    rows = []
    for r in results:
        kind, label = palette.get(r.status, ("info", r.status))
        detail = r.message
        if r.status in ("indexed", "replaced"):
            detail = f"{r.chunks} chunks from {r.chars:,} characters in {r.seconds:.1f} s. " + " ".join(r.warnings)
        rows.append(f'<div class="dm-log-row dm-note-{kind}"><b>{label}: {esc(r.filename)}</b>{esc(detail.strip())}</div>')
    return '<div class="dm-log">' + "".join(rows) + "</div>"


def model_status_html(status: dict) -> str:
    rows = [("Embedding model", status["embedding_model"] + (" (fallback)" if status["embedding_fallback"] else "")),
            ("Generation model", status["generation_model"] + (" (fallback)" if status["generation_fallback"] else "")
             + ("" if status["generator_loaded"] else " (loads on first question)")),
            ("Device", status["device"])]
    body = '<div class="dm-kv">' + "".join(f"<span>{k}</span><b>{esc(v)}</b>" for k, v in rows) + "</div>"
    warnings = "".join(status_html(w, "warn") for w in status["warnings"])
    return body + warnings


TYPING_HTML = '<div class="dm-typing" aria-label="DocuMind is thinking"><span></span><span></span><span></span></div>'


def error_html(message: str) -> str:
    return status_html(message, "error")


def _highlight(text: str, question: str) -> str:
    safe = esc(text)
    terms = {t.replace("'", "") for t in tokenize(question) if t not in STOPWORDS and len(t) > 2}
    terms -= {"amp", "quot", "lt", "gt"}
    if not terms:
        return safe
    pattern = r"(?i)(?<![&#\w])(" + "|".join(re.escape(t) for t in sorted(terms, key=len, reverse=True)) + r")"
    return re.sub(pattern, r"<mark>\1</mark>", safe)


def _source_html(source, question: str) -> str:
    page = f"Page {source.page}" if source.page else "Page n/a"
    width = max(0, min(100, round(source.score * 100)))
    badge = "dm-badge dm-badge-on" if source.cited else "dm-badge"
    title = "Used in the answer" if source.cited else "Retrieved but not used in the answer"
    return (f'<details class="dm-src"><summary><span class="{badge}" title="{title}">{source.number}</span>'
            f'<span class="dm-src-name">{esc(source.filename)}</span><span class="dm-chip">{page}</span>'
            f'<span class="dm-bar" title="Cosine similarity"><i style="width:{width}%"></i></span>'
            f'<span class="dm-score">similarity {source.score:.2f}</span></summary>'
            f'<div class="dm-passage">{_highlight(source.text, question)}</div></details>')


def render_answer(answer) -> str:
    """HTML for one assistant message: answer text, notes, metadata and expandable sources."""
    if answer.kind in ("invalid", "empty"):
        return status_html(answer.text, "warn")
    parts: list[str] = []
    if answer.kind == "answer":
        pills = "".join(f'<span class="dm-ref" title="Source {s.number}: {esc(s.filename)}">{s.number}</span>'
                        for s in answer.sources if s.cited)
        parts.append(f'<div class="dm-answer-text">{esc(answer.text).replace(chr(10), "<br>")} {pills}</div>')
        if answer.low_confidence:
            parts.append(status_html("Low confidence: this answer's wording barely overlaps with the sources. Check them below.", "warn"))
    else:
        parts.append(status_html(answer.text, "warn"))
    for note in answer.warnings:
        parts.append(status_html(note, "warn"))
    if answer.truncated:
        parts.append(status_html("Some retrieved text was shortened to fit the model's input limit.", "info"))
    if answer.sources:
        title = "Sources" if answer.kind == "answer" else "Closest passages (not used)"
        parts.append(f'<div class="dm-sources"><div class="dm-sources-title">{title}</div>'
                     + "".join(_source_html(s, answer.standalone_question or answer.question) for s in answer.sources) + "</div>")
    meta = [f"{len(answer.sources)} source{'s' if len(answer.sources) != 1 else ''}", f"{answer.seconds:.1f} s"]
    if answer.model:
        meta.append(answer.model.split("/")[-1])
    parts.append('<div class="dm-meta">' + "".join(f"<span>{esc(m)}</span>" for m in meta) + "</div>")
    return "".join(parts)


def how_it_works_html() -> str:
    def flow(*steps: str) -> str:
        return '<div class="dm-flow">' + "<i>then</i>".join(f"<span>{s}</span>" for s in steps) + "</div>"
    return ('<div class="dm-doc-text"><div class="dm-h">When you index documents</div>'
            + flow("Upload", "Extract text", "Clean", "Chunk with overlap", "Add metadata", "Embed", "Store in FAISS")
            + '<div class="dm-h">When you ask a question</div>'
            + flow("Embed the question", "Search FAISS", "Apply threshold", "Rerank by keywords", "Build context", "Generate answer", "Show citations")
            + "<p><b>Embeddings</b> turn text into numbers so that passages with similar meaning end up close together. "
              "<b>Chunks</b> are short passages; neighbouring chunks overlap so facts at a boundary are not lost. "
              "<b>FAISS</b> finds the closest chunks quickly. The language model only sees those chunks, and is told to answer from them alone "
              "or say that the answer is not in your documents.</p>"
              "<p>Each source shows its similarity score (higher means closer; 1.00 is identical). Filled numbers mark passages that support the answer.</p></div>")

In [ ]:
%%writefile ui/dashboard.py
"""Sidebar (upload, library, statistics), settings tab and their event handlers."""
from __future__ import annotations

import gradio as gr

from config import (ALLOWED_EXTENSIONS, EMBEDDING_MODELS, GENERATION_MODELS, MAX_FILE_MB, SAMPLE_DIR)
from rag.loaders import ocr_available
from ui.components import (hero_html, library_html, logo_html, model_status_html, settings_from_inputs, stats_html,
                           status_html, upload_log_html)

SYNC_KEYS = ("hero", "stats", "library", "delete_dd", "filter_dd", "model_status")


def snapshot(pipeline) -> tuple:
    """Fresh values for every component that mirrors repository state."""
    docs = pipeline.list_documents()
    stats = pipeline.stats()
    choices = [(d["filename"], d["doc_id"]) for d in docs]
    return (hero_html(stats), stats_html(stats), library_html(docs), gr.Dropdown(choices=choices, value=None),
            gr.Dropdown(choices=choices, value=[]), model_status_html(pipeline.status()))


def sync_outputs(c: dict) -> list:
    return [c[k] for k in SYNC_KEYS]


def build_sidebar(pipeline) -> dict:
    c: dict = {}
    stats, docs = pipeline.stats(), pipeline.list_documents()
    with gr.Column(scale=1, min_width=330, elem_classes="dm-side"):
        gr.HTML(logo_html())
        gr.HTML('<div class="dm-h">Upload documents</div>')
        c["files"] = gr.File(label=f"PDF, DOCX, TXT or Markdown, up to {MAX_FILE_MB} MB each", file_count="multiple",
                             file_types=list(ALLOWED_EXTENSIONS), type="filepath", height=130)
        with gr.Row():
            c["index_btn"] = gr.Button("Index documents", variant="primary")
            c["sample_btn"] = gr.Button("Load samples", variant="secondary")
        c["upload_log"] = gr.HTML(status_html("Choose files, then select Index documents.", "info"))
        gr.HTML('<div class="dm-h">Document library</div>')
        c["library"] = gr.HTML(library_html(docs))
        c["delete_dd"] = gr.Dropdown(choices=[(d["filename"], d["doc_id"]) for d in docs], value=None,
                                     label="Document to remove")
        c["delete_btn"] = gr.Button("Delete document", size="sm")
        gr.HTML('<div class="dm-h">Repository statistics</div>')
        c["stats"] = gr.HTML(stats_html(stats))
        with gr.Accordion("Clear repository", open=False):
            c["clear_confirm"] = gr.Checkbox(label="I want to remove all documents and vectors")
            c["clear_btn"] = gr.Button("Clear repository", variant="stop", size="sm")
    return c


def build_overview(pipeline) -> dict:
    return {"hero": gr.HTML(hero_html(pipeline.stats()))}


def build_settings(pipeline) -> dict:
    c: dict = {}
    status = pipeline.status()
    gr.HTML('<div class="dm-h">Chunking</div>')
    with gr.Row():
        c["chunk_size"] = gr.Slider(200, 2000, value=600, step=50, label="Chunk size (characters)",
                                    info="Applies to documents indexed from now on.")
        c["overlap"] = gr.Slider(0, 500, value=100, step=10, label="Chunk overlap (characters)",
                                 info="Characters shared by neighbouring chunks.")
    c["reindex_btn"] = gr.Button("Re-index all documents with these settings", size="sm")
    c["reindex_log"] = gr.HTML("")
    gr.HTML('<div class="dm-h">Retrieval and answers</div>')
    with gr.Row():
        c["top_k"] = gr.Slider(1, 10, value=4, step=1, label="Top-K chunks", info="How many passages the model sees.")
        c["threshold"] = gr.Slider(0, 0.9, value=0.25, step=0.01, label="Similarity threshold",
                                   info="Passages below this cosine similarity are ignored.")
    c["rerank"] = gr.Checkbox(value=True, label="Keyword reranking", info="Blend keyword coverage into the ranking.")
    c["detailed"] = gr.Checkbox(value=False, label="Detailed answers", info="Longer answers; slightly slower.")
    c["ocr"] = gr.Checkbox(value=False, label="OCR scanned PDFs", interactive=ocr_available(),
                           info="Needs Tesseract." if ocr_available() else "Unavailable: Tesseract is not installed.")
    gr.HTML('<div class="dm-h">Models</div>')
    with gr.Row():
        c["gen_model"] = gr.Dropdown(choices=[(f"{d}", n) for n, d in GENERATION_MODELS.items()], value=None,
                                     label="Generation model", allow_custom_value=True)
        c["emb_model"] = gr.Dropdown(choices=[(f"{d}", n) for n, d in EMBEDDING_MODELS.items()], value=None,
                                     label="Embedding model", allow_custom_value=True)
    c["load_models_btn"] = gr.Button("Load selected models", size="sm")
    c["model_status"] = gr.HTML(model_status_html(status))
    return c


def wire_dashboard(pipeline, c: dict) -> None:
    outs = sync_outputs(c)

    def _paths(files) -> list[str]:
        return [f if isinstance(f, str) else getattr(f, "name", str(f)) for f in (files or [])]

    def _run_ingest(paths, chunk_size, overlap, ocr, progress):
        try:
            settings = settings_from_inputs(chunk_size=chunk_size, overlap=overlap, ocr=ocr)
        except ValueError as exc:
            yield (status_html(str(exc), "error"), gr.update(), *snapshot(pipeline))
            return
        pipeline.busy = f"Indexing {len(paths)} file{'s' if len(paths) != 1 else ''}"
        yield (status_html(pipeline.busy + "...", "info"), gr.update(), *snapshot(pipeline))
        try:
            results = pipeline.ingest_files(paths, settings, progress=lambda f, t: progress(f, desc=t))
            log = upload_log_html(results)
        except Exception as exc:  # never leave the UI stuck in the busy state
            log = status_html(f"Indexing failed: {type(exc).__name__}: {exc}", "error")
        finally:
            pipeline.busy = ""
        yield (log, gr.File(value=None), *snapshot(pipeline))

    def on_index(files, chunk_size, overlap, ocr, progress=gr.Progress()):
        paths = _paths(files)
        if not paths:
            yield (status_html("Choose at least one file first.", "warn"), gr.update(), *snapshot(pipeline))
            return
        yield from _run_ingest(paths, chunk_size, overlap, ocr, progress)

    def on_samples(chunk_size, overlap, ocr, progress=gr.Progress()):
        paths = sorted(str(p) for p in SAMPLE_DIR.glob("*") if p.suffix.lower() in ALLOWED_EXTENSIONS)
        if not paths:
            yield (status_html("The sample_docs folder is empty.", "warn"), gr.update(), *snapshot(pipeline))
            return
        yield from _run_ingest(paths, chunk_size, overlap, ocr, progress)

    index_inputs = [c["files"], c["chunk_size"], c["overlap"], c["ocr"]]
    c["index_btn"].click(on_index, index_inputs, [c["upload_log"], c["files"], *outs], api_name="index_documents")
    c["sample_btn"].click(on_samples, index_inputs[1:], [c["upload_log"], c["files"], *outs], api_name="load_samples")

    def on_delete(doc_id):
        if not doc_id:
            return (status_html("Select a document to remove first.", "warn"), *snapshot(pipeline))
        name = pipeline.store.documents.get(doc_id, {}).get("filename", "document")
        ok = pipeline.delete_document(doc_id)
        message = status_html(f"Removed {name} and its vectors.", "ok") if ok else status_html("That document no longer exists.", "warn")
        return (message, *snapshot(pipeline))

    c["delete_btn"].click(on_delete, [c["delete_dd"]], [c["upload_log"], *outs], api_name="delete_document")

    def on_clear(confirm):
        if not confirm:
            return (status_html("Tick the confirmation box to clear the repository.", "warn"), gr.update(), *snapshot(pipeline))
        count = pipeline.clear_repository()
        return (status_html(f"Cleared the repository ({count} document{'s' if count != 1 else ''} removed).", "ok"),
                gr.Checkbox(value=False), *snapshot(pipeline))

    c["clear_btn"].click(on_clear, [c["clear_confirm"]], [c["upload_log"], c["clear_confirm"], *outs], api_name="clear_repository")

    def on_reindex(chunk_size, overlap, ocr, progress=gr.Progress()):
        try:
            settings = settings_from_inputs(chunk_size=chunk_size, overlap=overlap, ocr=ocr)
        except ValueError as exc:
            return (status_html(str(exc), "error"), *snapshot(pipeline))
        if not pipeline.list_documents():
            return (status_html("There are no documents to re-index.", "warn"), *snapshot(pipeline))
        pipeline.busy = "Re-indexing"
        try:
            results = pipeline.reindex_all(settings, progress=lambda f, t: progress(f, desc=t))
            log = upload_log_html(results)
        finally:
            pipeline.busy = ""
        return (log, *snapshot(pipeline))

    c["reindex_btn"].click(on_reindex, [c["chunk_size"], c["overlap"], c["ocr"]], [c["reindex_log"], *outs], api_name="reindex")

    def on_models(gen_model, emb_model, chunk_size, overlap, progress=gr.Progress()):
        messages = []
        pipeline.busy = "Loading models"
        try:
            if emb_model:
                progress(0.2, desc="Loading embedding model")
                settings = settings_from_inputs(chunk_size=chunk_size, overlap=overlap)
                messages.append(pipeline.set_embedding_model(emb_model, settings))
            if gen_model:
                progress(0.6, desc="Loading generation model")
                messages.append(pipeline.set_generation_model(gen_model))
        except Exception as exc:
            messages.append(f"Model switch failed: {type(exc).__name__}: {exc}")
        finally:
            pipeline.busy = ""
        if not messages:
            messages = ["Pick a model first."]
        return (*snapshot(pipeline)[:-1], model_status_html(pipeline.status()) + "".join(status_html(m, "info") for m in messages))

    c["load_models_btn"].click(on_models, [c["gen_model"], c["emb_model"], c["chunk_size"], c["overlap"]], outs, api_name="load_models")

In [ ]:
%%writefile ui/chat.py
"""Chat workspace: conversation, input, suggested prompts and document filter."""
from __future__ import annotations

import inspect

import gradio as gr

from ui.components import SUGGESTIONS, TYPING_HTML, error_html, render_answer, settings_from_inputs

_CHATBOT_KWARGS = {"type": "messages"} if "type" in inspect.signature(gr.Chatbot.__init__).parameters else {}
PLACEHOLDER = ("<div style='text-align:center;padding:32px 12px'><b style='font-size:17px'>Ask anything about your documents</b>"
               "<p style='color:#667085;margin-top:6px'>Index some files first, or load the samples, then ask a question below.</p></div>")


def message_text(content) -> str:
    """Plain text of a chat message (Gradio 6 passes lists of content blocks, Gradio 5 plain strings)."""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return str(content.get("text", "")) if content.get("type", "text") == "text" else ""
    if isinstance(content, (list, tuple)):
        return " ".join(t for t in (message_text(part) for part in content) if t)
    return ""


def build_chat(pipeline) -> dict:
    c: dict = {}
    c["chatbot"] = gr.Chatbot(value=[], height=520, show_label=False, placeholder=PLACEHOLDER,
                              elem_classes="dm-chatbot", sanitize_html=True, **_CHATBOT_KWARGS)
    with gr.Row():
        c["msg"] = gr.Textbox(placeholder="Ask a question about your documents", show_label=False, container=False,
                              lines=1, max_lines=4, scale=8)
        c["send"] = gr.Button("Send", variant="primary", scale=1, min_width=90)
    with gr.Row():
        c["filter_dd"] = gr.Dropdown(choices=[], value=[], multiselect=True, scale=3,
                                     label="Search only in these documents (leave empty to search all)")
        c["clear_chat"] = gr.Button("Clear conversation", size="sm", scale=1, min_width=150)
    gr.HTML('<div class="dm-sugg-title">Suggested questions (they match the sample documents)</div>')
    with gr.Row():
        c["suggestions"] = [gr.Button(q, size="sm", elem_classes="dm-chip-btn") for q in SUGGESTIONS]
    return c


def wire_chat(pipeline, c: dict) -> None:
    def respond(message, history, top_k, threshold, rerank, detailed, doc_ids):
        history = list(history or [])
        message = (message or "").strip()
        if not message:
            gr.Warning("Type a question first.")
            yield history, ""
            return
        user = {"role": "user", "content": message}
        yield history + [user, {"role": "assistant", "content": TYPING_HTML}], ""
        try:
            settings = settings_from_inputs(top_k=top_k, threshold=threshold, rerank=rerank, detailed=detailed,
                                            doc_ids=doc_ids)
            previous = [message_text(m.get("content")) for m in history if m.get("role") == "user"]
            previous = [q for q in previous if q.strip()]
            body = render_answer(pipeline.ask(message, settings, previous))
        except Exception as exc:  # surface any failure in the chat instead of a silent crash
            body = error_html(f"Something went wrong: {type(exc).__name__}: {exc}")
        yield history + [user, {"role": "assistant", "content": body}], ""

    inputs = [c["msg"], c["chatbot"], c["top_k"], c["threshold"], c["rerank"], c["detailed"], c["filter_dd"]]
    outputs = [c["chatbot"], c["msg"]]
    c["msg"].submit(respond, inputs, outputs, api_name="ask")
    c["send"].click(respond, inputs, outputs, api_name=False)
    c["clear_chat"].click(lambda: ([], ""), None, outputs, api_name=False)

    def prefill(text):
        def setter():
            return text
        return setter

    for button, question in zip(c["suggestions"], SUGGESTIONS):
        button.click(prefill(question), None, c["msg"], api_name=False).then(respond, inputs, outputs, api_name=False)

In [ ]:
%%writefile app.py
"""DocuMind entry point.  Run locally with `python app.py` (add --share for a public link)."""
from __future__ import annotations

import argparse
import inspect

import gradio as gr

from config import DEFAULT_EMBEDDING_MODEL, DEFAULT_GENERATION_MODEL
from rag.pipeline import DocuMindPipeline
from ui.chat import build_chat, wire_chat
from ui.components import CSS, FORCE_LIGHT_JS, how_it_works_html, make_theme
from ui.dashboard import build_overview, build_settings, build_sidebar, snapshot, sync_outputs, wire_dashboard

# Gradio 6 takes theme/css/js in launch(); Gradio 5 takes them in Blocks().
_STYLE_IN_LAUNCH = "css" in inspect.signature(gr.Blocks.launch).parameters


def build_app(pipeline: DocuMindPipeline | None = None) -> gr.Blocks:
    pipeline = pipeline or DocuMindPipeline()
    style = {} if _STYLE_IN_LAUNCH else {"theme": make_theme(), "css": CSS, "js": FORCE_LIGHT_JS}
    with gr.Blocks(title="DocuMind", **style) as demo:
        c: dict = {}
        with gr.Row(elem_classes="dm-layout", equal_height=False):
            c.update(build_sidebar(pipeline))
            with gr.Column(scale=3, min_width=480):
                c.update(build_overview(pipeline))
                with gr.Tabs():
                    with gr.Tab("Chat"):
                        c.update(build_chat(pipeline))
                    with gr.Tab("Settings"):
                        c.update(build_settings(pipeline))
                    with gr.Tab("How it works"):
                        gr.HTML(how_it_works_html())
        wire_dashboard(pipeline, c)
        wire_chat(pipeline, c)
        demo.load(lambda: snapshot(pipeline), None, sync_outputs(c), api_name=False)
    demo.pipeline = pipeline
    return demo


def launch_app(demo: gr.Blocks, share: bool = False, **kwargs):
    style = {"theme": make_theme(), "css": CSS, "js": FORCE_LIGHT_JS} if _STYLE_IN_LAUNCH else {}
    demo.queue(max_size=32)
    return demo.launch(share=share, **style, **kwargs)


def main() -> None:
    parser = argparse.ArgumentParser(description="DocuMind: RAG document assistant")
    parser.add_argument("--share", action="store_true", help="create a public Gradio link")
    parser.add_argument("--port", type=int, default=7860)
    parser.add_argument("--embedding-model", default=DEFAULT_EMBEDDING_MODEL)
    parser.add_argument("--generation-model", default=DEFAULT_GENERATION_MODEL)
    parser.add_argument("--no-preload", action="store_true", help="load the LLM on the first question")
    args = parser.parse_args()
    pipeline = DocuMindPipeline(embedding_model=args.embedding_model, generation_model=args.generation_model)
    if not args.no_preload:
        for warning in pipeline.load_models():
            print("WARNING:", warning)
    launch_app(build_app(pipeline), share=args.share, server_port=args.port)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile requirements.txt
gradio>=6.0,<7
faiss-cpu>=1.8
sentence-transformers>=3.0
transformers>=4.44
sentencepiece
torch>=2.1
numpy>=1.24
pypdf>=4.0
python-docx>=1.1
pillow
pytesseract
pytest>=8.0

In [ ]:
%%writefile pytest.ini
[pytest]
pythonpath = .
testpaths = tests
filterwarnings =
    ignore::DeprecationWarning
    ignore::UserWarning

In [ ]:
%%writefile tests/conftest.py
import sys
from pathlib import Path

import pytest

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

from config import RAGSettings  # noqa: E402
from rag.embeddings import HashingEmbedder  # noqa: E402
from rag.generator import ExtractiveGenerator  # noqa: E402
from rag.pipeline import DocuMindPipeline  # noqa: E402
from scripts.make_samples import main as make_samples  # noqa: E402


@pytest.fixture(scope="session")
def sample_dir(tmp_path_factory) -> Path:
    """The sample repository (MD, DOCX, TXT, PDF) generated into a temp folder."""
    return make_samples(tmp_path_factory.mktemp("samples"))


@pytest.fixture
def sample_files(sample_dir) -> dict:
    return {p.name: p for p in sorted(sample_dir.iterdir())}


@pytest.fixture
def settings() -> RAGSettings:
    # The hashing embedder gives lower cosine scores than MiniLM, so tests use a lower threshold.
    return RAGSettings(similarity_threshold=0.12)


@pytest.fixture
def embedder() -> HashingEmbedder:
    return HashingEmbedder()


@pytest.fixture
def make_pipeline(tmp_path):
    def factory(name: str = "data") -> DocuMindPipeline:
        return DocuMindPipeline(tmp_path / name, embedder=HashingEmbedder(), generator=ExtractiveGenerator())
    return factory


@pytest.fixture
def pipeline(make_pipeline) -> DocuMindPipeline:
    return make_pipeline()


@pytest.fixture
def loaded_pipeline(pipeline, sample_files, settings) -> DocuMindPipeline:
    results = pipeline.ingest_files(list(sample_files.values()), settings)
    assert all(r.status == "indexed" for r in results), results
    return pipeline

In [ ]:
%%writefile tests/pdf_utils.py
"""Minimal dependency-free PDF writer (text only) used by the tests and the sample generator."""
from __future__ import annotations

import textwrap
from pathlib import Path


def _escape(line: str) -> bytes:
    return line.encode("latin-1", "replace").replace(b"\\", b"\\\\").replace(b"(", b"\\(").replace(b")", b"\\)")


def make_pdf(pages: list[str], path: str | Path) -> Path:
    """Write one PDF page per string. Paragraphs are separated by blank lines."""
    objects: list[bytes] = [b"<< /Type /Catalog /Pages 2 0 R >>"]
    kids = " ".join(f"{4 + 2 * i} 0 R" for i in range(len(pages)))
    objects.append(f"<< /Type /Pages /Kids [{kids}] /Count {len(pages)} >>".encode())
    objects.append(b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>")
    for i, text in enumerate(pages):
        lines: list[str] = []
        for paragraph in text.strip().split("\n\n"):
            lines.extend(textwrap.wrap(" ".join(paragraph.split()), 88) or [""])
            lines.append("")
        stream = b"BT /F1 11 Tf 15 TL 50 740 Td\n" + b"".join(b"(" + _escape(l) + b") Tj T*\n" for l in lines) + b"ET"
        objects.append((f"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Resources << /Font << /F1 3 0 R >> >> "
                        f"/Contents {5 + 2 * i} 0 R >>").encode())
        objects.append(b"<< /Length %d >>\nstream\n" % len(stream) + stream + b"\nendstream")
    out = bytearray(b"%PDF-1.4\n")
    offsets = []
    for number, body in enumerate(objects, start=1):
        offsets.append(len(out))
        out += f"{number} 0 obj\n".encode() + body + b"\nendobj\n"
    xref = len(out)
    out += f"xref\n0 {len(objects) + 1}\n0000000000 65535 f \n".encode()
    for offset in offsets:
        out += f"{offset:010d} 00000 n \n".encode()
    out += f"trailer\n<< /Size {len(objects) + 1} /Root 1 0 R >>\nstartxref\n{xref}\n%%EOF\n".encode()
    path = Path(path)
    path.write_bytes(bytes(out))
    return path

In [ ]:
%%writefile tests/test_loaders.py
import pytest
from pypdf import PdfWriter

from rag.loaders import DocumentError, clean_text, load_document, ocr_available, validate_file
from tests.pdf_utils import make_pdf


def test_txt_extraction(sample_files):
    doc = load_document(sample_files["03_renewable_energy_overview.txt"])
    assert doc.file_type == "txt" and doc.pages[0].number is None
    assert "capacity factor" in doc.pages[0].text and doc.size_bytes > 0 and len(doc.sha256) == 64


def test_markdown_syntax_is_stripped(sample_files):
    text = load_document(sample_files["01_intro_to_machine_learning.md"]).pages[0].text
    assert "Supervised learning uses labeled examples" in text
    assert "**" not in text and "# " not in text


def test_docx_extracts_paragraphs_and_tables(sample_files):
    text = load_document(sample_files["02_remote_work_policy.docx"]).pages[0].text
    assert "90-day probation" in text
    assert "Expense: Home-office stipend; Amount: $500; Frequency: One time." in text


def test_pdf_extraction_keeps_page_numbers(sample_files):
    doc = load_document(sample_files["04_photosynthesis_and_plant_biology.pdf"])
    assert [p.number for p in doc.pages] == [1, 2, 3] and doc.total_pages == 3
    assert "stroma" in doc.pages[1].text and "limiting factor" in doc.pages[2].text


def test_clean_text_rules():
    assert clean_text("exam-\nple of  a   hyphen") == "example of a hyphen"
    assert clean_text("line one\nline two\n\n\n\nnew paragraph") == "line one line two\n\nnew paragraph"
    assert clean_text("a\x00b\x07c\u00a0d") == "a b c d"
    assert clean_text("- item one\n- item two") == "- item one\n- item two"
    assert clean_text("\ufb01ne") == "fine"  # ligature normalised
    assert clean_text("") == "" and clean_text("   \n ") == ""


def test_cp1252_text_is_decoded(tmp_path):
    path = tmp_path / "latin.txt"
    path.write_bytes("Caf\xe9 au lait is a popular coffee drink in many countries.".encode("cp1252"))
    assert "Café au lait" in load_document(path).pages[0].text


@pytest.mark.parametrize("name,content,message", [
    ("empty.txt", b"", "empty"),
    ("blank.txt", b"   \n\n  ", "no readable text"),
    ("data.exe", b"MZ....", "Unsupported"),
    ("noext", b"hello", "Unsupported"),
    ("fake.pdf", b"this is not a pdf", "PDF"),
    ("fake.docx", b"this is not a zip", "DOCX"),
    ("binary.txt", b"abc\x00\x01\x02\x00" * 50, "binary"),
])
def test_invalid_files_are_rejected_gracefully(tmp_path, name, content, message):
    path = tmp_path / name
    path.write_bytes(content)
    with pytest.raises(DocumentError) as err:
        load_document(path)
    assert message.lower() in str(err.value).lower()


def test_corrupted_pdf_with_valid_header(tmp_path):
    path = tmp_path / "broken.pdf"
    path.write_bytes(b"%PDF-1.4\n1 0 obj << /Broken >> endobj\ngarbage garbage")
    with pytest.raises(DocumentError):
        load_document(path)


def test_corrupted_docx_zip(tmp_path):
    import zipfile
    path = tmp_path / "broken.docx"
    with zipfile.ZipFile(path, "w") as z:
        z.writestr("hello.txt", "not a word file")
    with pytest.raises(DocumentError):
        load_document(path)


def test_pdf_without_text_and_encrypted_pdf(tmp_path):
    blank = tmp_path / "blank.pdf"
    writer = PdfWriter()
    writer.add_blank_page(200, 200)
    with open(blank, "wb") as fh:
        writer.write(fh)
    with pytest.raises(DocumentError, match="No extractable text"):
        load_document(blank)
    locked = tmp_path / "locked.pdf"
    writer.encrypt("secret")
    with open(locked, "wb") as fh:
        writer.write(fh)
    with pytest.raises(DocumentError, match="password"):
        load_document(locked)


def test_file_size_limit(tmp_path):
    path = tmp_path / "big.txt"
    path.write_text("word " * 5000)
    with pytest.raises(DocumentError, match="limit"):
        load_document(path, max_mb=0.001)
    assert validate_file(path) == ".txt"


def test_missing_file():
    with pytest.raises(DocumentError, match="not found"):
        load_document("/nonexistent/file.txt")


@pytest.mark.skipif(not ocr_available(), reason="Tesseract not installed")
def test_ocr_reads_scanned_pdf(tmp_path):
    from PIL import Image, ImageDraw, ImageFont

    image = Image.new("RGB", (1000, 220), "white")
    ImageDraw.Draw(image).text((30, 60), "Quarterly revenue grew strongly", fill="black", font=ImageFont.load_default(size=60))
    path = tmp_path / "scan.pdf"
    image.save(path, "PDF")
    with pytest.raises(DocumentError, match="No extractable text"):
        load_document(path, use_ocr=False)
    doc = load_document(path, use_ocr=True)
    assert "revenue" in doc.pages[0].text.lower() and any("OCR" in w for w in doc.warnings)

In [ ]:
%%writefile tests/test_chunking.py
import pytest

from rag.chunking import chunk_pages, chunk_text, split_sentences
from rag.loaders import Page

TEXT = " ".join(f"Sentence number {i} talks about topic {i % 4} in some detail." for i in range(60))


def test_chunks_respect_size_and_cover_all_text():
    chunks = chunk_text(TEXT, 300, 80)
    assert len(chunks) > 5 and all(len(c) <= 300 for c in chunks)
    joined = " ".join(chunks)
    assert all(f"Sentence number {i} " in joined for i in range(60))


def test_neighbouring_chunks_overlap():
    chunks = chunk_text(TEXT, 300, 100)
    for left, right in zip(chunks, chunks[1:]):
        assert left.split(". ")[-1][:20] in right  # last sentence of left reappears in right


def test_zero_overlap_has_no_repetition():
    chunks = chunk_text(TEXT, 300, 0)
    assert sum(len(c) for c in chunks) <= len(TEXT)
    assert len(set(chunks)) == len(chunks)


@pytest.mark.parametrize("size,overlap", [(50, 0), (300, 300), (300, -1), (200, 250)])
def test_invalid_parameters(size, overlap):
    with pytest.raises(ValueError):
        chunk_text(TEXT, size, overlap)


def test_long_sentence_is_hard_split():
    chunks = chunk_text("word " * 400 + "end.", 200, 20)
    assert len(chunks) > 5 and all(len(c) <= 200 for c in chunks)
    assert all(len(c) <= 200 for c in chunk_text("x" * 1000, 200, 20))


def test_empty_text_gives_no_chunks():
    assert chunk_text("", 300, 50) == [] and chunk_text("   \n\n ", 300, 50) == []


def test_sentence_splitter_keeps_paragraphs_apart():
    assert split_sentences("First one. Second one.\n\nThird one!") == ["First one.", "Second one.", "Third one!"]


def test_page_numbers_and_indices_are_preserved():
    pages = [Page(1, TEXT[:600]), Page(2, TEXT[600:1400]), Page(None, TEXT[1400:])]
    chunks = chunk_pages(pages, 250, 50)
    assert [c.chunk_index for c in chunks] == list(range(len(chunks)))
    assert {c.page for c in chunks} == {1, 2, None}
    assert [c.page for c in chunks] == sorted((c.page for c in chunks), key=lambda p: (p is None, p or 0))


def test_tiny_fragments_are_dropped():
    assert chunk_pages([Page(1, "12"), Page(2, "-- ..")], 200, 20) == []

In [ ]:
%%writefile tests/test_embeddings.py
import numpy as np

from rag import embeddings
from rag.embeddings import HashingEmbedder, load_embedder

TEXTS = ["The cat sat on the mat.", "Dogs love long walks outside.", "Quarterly revenue grew by ten percent.", "The cat sat on the mat."]


def test_shape_dtype_and_normalisation(embedder):
    vectors = embedder.encode(TEXTS)
    assert vectors.shape == (4, embedder.dim) and vectors.dtype == np.float32
    assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-5)
    assert np.allclose(vectors[0], vectors[3])


def test_batch_size_does_not_change_results():
    a = HashingEmbedder().encode(TEXTS, batch_size=1)
    b = HashingEmbedder().encode(TEXTS, batch_size=64)
    assert np.allclose(a, b)


def test_cache_avoids_regeneration(embedder):
    embedder.encode(TEXTS)
    assert embedder.generated == 3  # the duplicate text is only embedded once
    embedder.encode(TEXTS)
    assert embedder.generated == 3 and embedder.cache_hits >= 4


def test_progress_callback_and_empty_input(embedder):
    seen = []
    embedder.encode(TEXTS, batch_size=2, progress=lambda done, total: seen.append((done, total)))
    assert seen[-1] == (3, 3)
    assert embedder.encode([]).shape == (0, embedder.dim)


def test_similar_text_scores_higher(embedder):
    q, near, far = embedder.encode(["How much is the stipend?", "The stipend is $500 for remote staff.", "Volcanoes erupt lava."])
    assert float(q @ near) > float(q @ far)


def test_loader_falls_back_when_model_cannot_load(monkeypatch):
    def boom(self):
        raise OSError("no network")
    monkeypatch.setattr(embeddings.SentenceTransformerEmbedder, "load", boom)
    embedder, warning = load_embedder("some/model")
    assert embedder.is_fallback and "fallback" in warning.lower()

In [ ]:
%%writefile tests/test_vectorstore.py
import json

import numpy as np
import pytest

from rag.vectorstore import META_FILE, VectorStore, VectorStoreError

DIM = 8


def unit(*values):
    vec = np.zeros(DIM, dtype=np.float32)
    vec[: len(values)] = values
    return vec / np.linalg.norm(vec)


def add(store, doc_id, name, vectors):
    chunks = [{"page": i + 1, "chunk_index": i, "text": f"{name} chunk {i}"} for i in range(len(vectors))]
    store.add_document({"doc_id": doc_id, "filename": name, "added_at": f"2024-01-0{len(store.documents) + 1}"}, chunks, np.stack(vectors))


@pytest.fixture
def store(tmp_path):
    s = VectorStore(tmp_path / "db", DIM, "test-model")
    add(s, "d1", "a.txt", [unit(1, 0), unit(0.9, 0.1)])
    add(s, "d2", "b.txt", [unit(0, 1), unit(0.1, 0.9)])
    return s


def test_search_returns_best_first_with_metadata(store):
    hits = store.search(unit(1, 0.05), 3)
    assert hits[0][0]["filename"] == "a.txt" and hits[0][1] >= hits[1][1] >= hits[2][1]
    assert {"id", "doc_id", "page", "chunk_index", "text"} <= set(hits[0][0])


def test_document_filter(store):
    hits = store.search(unit(1, 0), 4, doc_ids={"d2"})
    assert hits and all(h[0]["doc_id"] == "d2" for h in hits)


def test_delete_removes_vectors_and_metadata(store):
    assert store.vector_count == 4
    assert store.delete_document("d1") == 2
    assert store.vector_count == 2 and len(store.chunks) == 2 and "d1" not in store.documents
    assert all(h[0]["doc_id"] == "d2" for h in store.search(unit(1, 0), 10))
    assert store.delete_document("d1") == 0


def test_clear(store):
    store.clear()
    assert store.vector_count == 0 and not store.documents and store.search(unit(1), 3) == []


def test_persistence_round_trip(store, tmp_path):
    reloaded = VectorStore(tmp_path / "db", DIM, "test-model")
    assert reloaded.vector_count == 4 and set(reloaded.documents) == {"d1", "d2"}
    assert reloaded.search(unit(0, 1), 1)[0][0]["filename"] == "b.txt"
    add(reloaded, "d3", "c.txt", [unit(0, 0, 1)])
    assert reloaded.next_id == 6  # ids keep growing: no id is ever reused


def test_duplicate_and_bad_shapes_are_rejected(store):
    with pytest.raises(VectorStoreError, match="already"):
        add(store, "d1", "a.txt", [unit(1)])
    with pytest.raises(VectorStoreError, match="shape"):
        store.add_document({"doc_id": "x", "filename": "x"}, [{"page": 1, "chunk_index": 0, "text": "t"}], np.zeros((2, DIM), dtype=np.float32))
    with pytest.raises(VectorStoreError, match="dims"):
        store.search(np.ones(3, dtype=np.float32), 2)


def test_stale_records_are_reconciled_on_load(store, tmp_path):
    meta_path = tmp_path / "db" / META_FILE
    meta = json.loads(meta_path.read_text())
    meta["chunks"].pop(next(iter(meta["chunks"])))  # metadata lost for one vector
    meta_path.write_text(json.dumps(meta))
    reloaded = VectorStore(tmp_path / "db", DIM, "test-model")
    assert reloaded.vector_count == 3 and len(reloaded.chunks) == 3 and reloaded.load_warning


def test_corrupted_files_reset_the_store(store, tmp_path):
    (tmp_path / "db" / "index.faiss").write_bytes(b"garbage")
    reloaded = VectorStore(tmp_path / "db", DIM, "test-model")
    assert reloaded.vector_count == 0 and "unreadable" in reloaded.load_warning


def test_model_change_exposes_stale_documents(store, tmp_path):
    other = VectorStore(tmp_path / "db", DIM, "another-model")
    assert other.vector_count == 0 and len(other.stale_documents) == 2

In [ ]:
%%writefile tests/test_retrieval.py
import pytest

from rag.retrieval import Retriever, contextualize_question

QUESTIONS = [
    ("How much is the home-office stipend?", "02_remote_work_policy.docx"),
    ("Where does the Calvin cycle take place?", "04_photosynthesis_and_plant_biology.pdf"),
    ("What is overfitting?", "01_intro_to_machine_learning.md"),
    ("What is the capacity factor of offshore wind farms?", "03_renewable_energy_overview.txt"),
    ("What does a vector database such as FAISS do?", "05_rag_overview.md"),
]


@pytest.mark.parametrize("question,expected", QUESTIONS)
def test_relevant_document_is_ranked_first(loaded_pipeline, question, expected):
    results = loaded_pipeline.retriever.retrieve(question, top_k=3, threshold=0.1)
    assert results and results[0].filename == expected
    assert [r.rank for r in results] == list(range(1, len(results) + 1))


def test_source_metadata_is_preserved(loaded_pipeline):
    top = loaded_pipeline.retriever.retrieve("Where does the Calvin cycle take place?", top_k=1, threshold=0.1)[0]
    assert top.page == 2 and "stroma" in top.text and 0 < top.score <= 1 and top.doc_id and top.chunk_id > 0


def test_top_k_is_respected(loaded_pipeline):
    assert len(loaded_pipeline.retriever.retrieve("energy", top_k=2, threshold=0.0)) <= 2
    assert len(loaded_pipeline.retriever.retrieve("energy", top_k=6, threshold=0.0)) <= 6


def test_threshold_filters_irrelevant_chunks(loaded_pipeline):
    question = "Who won the football world cup in 2018?"
    assert loaded_pipeline.retriever.retrieve(question, top_k=4, threshold=0.5) == []
    assert all(r.score >= 0.3 for r in loaded_pipeline.retriever.retrieve("How much is the stipend?", top_k=4, threshold=0.3))


def test_rerank_blends_keyword_coverage(loaded_pipeline):
    plain = loaded_pipeline.retriever.retrieve("home office stipend amount", 3, 0.0, rerank=False)
    mixed = loaded_pipeline.retriever.retrieve("home office stipend amount", 3, 0.0, rerank=True)
    assert all(abs(r.final_score - r.score) < 1e-9 for r in plain)
    assert any(abs(r.final_score - r.score) > 1e-6 for r in mixed) and mixed[0].keyword_score > 0


def test_document_filter(loaded_pipeline):
    pdf = next(d for d in loaded_pipeline.list_documents() if d["filename"].endswith(".pdf"))
    results = loaded_pipeline.retriever.retrieve("energy", top_k=4, threshold=0.0, doc_ids={pdf["doc_id"]})
    assert results and {r.filename for r in results} == {pdf["filename"]}


def test_empty_query_and_empty_store(loaded_pipeline, make_pipeline):
    with pytest.raises(ValueError):
        loaded_pipeline.retriever.retrieve("   ")
    assert make_pipeline("empty").retriever.retrieve("anything", 3, 0.0) == []


def test_follow_up_questions_are_made_standalone():
    history = ["How much is the home-office stipend?"]
    assert "regarding: How much is the home-office stipend?" in contextualize_question("What about the internet?", history)
    assert "regarding" in contextualize_question("Is it taxable?", history)
    full = "Which days are the anchor days in the office?"
    assert contextualize_question(full, history) == full
    assert contextualize_question("What about it?", []) == "What about it?"

In [ ]:
%%writefile tests/test_generator.py
import os
from types import SimpleNamespace

import pytest

from rag.generator import ExtractiveGenerator, build_prompt, find_supporting_chunks, is_not_found


def count(text):
    return len(text.split())


def truncate(text, n):
    return " ".join(text.split()[:n])


@pytest.mark.parametrize("text", ["", "unanswerable", "Unanswerable.", "I don't know", "None", "not mentioned"])
def test_not_found_phrases(text):
    assert is_not_found(text)


def test_real_answers_are_not_flagged():
    assert not is_not_found("$500 one time") and not is_not_found("Yes")


def test_prompt_contains_grounding_instruction_and_numbered_context():
    prompt, used, truncated = build_prompt("What is X?", ["alpha text", "beta text"], count, truncate, 500)
    assert "using only the information in the context" in prompt and "unanswerable" in prompt
    assert "[1] alpha text" in prompt and "[2] beta text" in prompt and prompt.rstrip().endswith("sentences:")
    assert used == 2 and not truncated


def test_prompt_respects_token_budget_and_keeps_question():
    texts = ["word " * 200, "second " * 200, "third " * 200]
    prompt, used, truncated = build_prompt("What is the answer to everything?", texts, count, truncate, 300)
    assert count(prompt) <= 300 and truncated and used < 3
    assert "What is the answer to everything?" in prompt and "[1]" in prompt


def test_supporting_chunks_follow_answer_wording():
    texts = ["The stipend is five hundred dollars paid once.", "Volcanoes erupt molten lava.", "Core hours are ten to three."]
    cited, coverage = find_supporting_chunks("A stipend of five hundred dollars.", texts)
    assert cited == [1] and coverage > 0.5
    cited, coverage = find_supporting_chunks("Completely unrelated zebra galaxy", texts)
    assert coverage == 0.0 and cited == [1]
    assert 2 in find_supporting_chunks("See [2] for details", texts)[0]


def chunks(*texts):
    return [SimpleNamespace(text=t) for t in texts]


def test_extractive_generator_is_grounded():
    ctx = chunks("Employees receive a one-time stipend of $500. Core hours are 10:00 to 15:00.", "Volcanoes erupt lava.")
    result = ExtractiveGenerator().generate("How large is the stipend?", ctx)
    assert result.found and "$500" in result.answer and "Volcanoes" not in result.answer


def test_extractive_generator_admits_missing_information():
    assert not ExtractiveGenerator().generate("Who painted the Mona Lisa?", chunks("Employees receive a stipend.")).found
    assert not ExtractiveGenerator().generate("???", chunks("Employees receive a stipend.")).found


@pytest.mark.skipif(os.environ.get("DOCUMIND_RUN_MODEL_TESTS") != "1", reason="set DOCUMIND_RUN_MODEL_TESTS=1 to download and test FLAN-T5")
def test_flan_t5_answers_from_context_and_declines_otherwise():
    from rag.generator import FlanT5Generator

    gen = FlanT5Generator("google/flan-t5-small", device="cpu").load()
    ctx = chunks("Remote employees receive a one-time home-office stipend of $500 and a monthly internet reimbursement of $40.")
    assert "500" in gen.generate("How much is the home-office stipend?", ctx).answer
    assert not gen.generate("What is the capital of France?", chunks("Volcanoes erupt molten lava from the earth.")).found

In [ ]:
%%writefile tests/test_pipeline.py
import shutil

import pytest

from config import NOT_FOUND_MESSAGE, RAGSettings
from rag.pipeline import safe_filename


def test_multiple_formats_are_indexed(loaded_pipeline):
    docs = loaded_pipeline.list_documents()
    assert {d["file_type"] for d in docs} == {"md", "docx", "txt", "pdf"} and len(docs) == 5
    stats = loaded_pipeline.stats()
    assert stats["documents"] == 5 and stats["chunks"] == stats["vectors"] > 5 and stats["embeddings_generated"] > 0
    pdf = next(d for d in docs if d["file_type"] == "pdf")
    assert pdf["pages"] == 3 and pdf["chunks"] >= 3 and pdf["size_bytes"] > 0


def test_duplicate_content_is_not_indexed_twice(pipeline, sample_files, settings, tmp_path):
    first = pipeline.ingest_file(sample_files["03_renewable_energy_overview.txt"], settings)
    copy = tmp_path / "copy_of_energy.txt"
    shutil.copy(sample_files["03_renewable_energy_overview.txt"], copy)
    second = pipeline.ingest_file(copy, settings)
    assert first.status == "indexed" and second.status == "duplicate" and "03_renewable" in second.message
    assert pipeline.stats()["documents"] == 1 and pipeline.stats()["vectors"] == first.chunks


def test_same_filename_new_content_replaces_old_version(pipeline, tmp_path, settings):
    path = tmp_path / "notes.txt"
    path.write_text("The first version talks about apples and orchards in great detail every autumn.")
    old = pipeline.ingest_file(path, settings)
    path.write_text("The second version talks about bananas and plantations in great detail every winter.")
    new = pipeline.ingest_file(path, settings)
    assert new.status == "replaced" and new.doc_id != old.doc_id
    docs = pipeline.list_documents()
    assert len(docs) == 1 and docs[0]["doc_id"] == new.doc_id
    assert all("apples" not in c["text"] for c in pipeline.store.chunks.values())
    assert not (pipeline.upload_dir / old_stored(old, "notes.txt")).exists()


def old_stored(result, name):
    return f"{result.doc_id}_{safe_filename(name)}"


def test_bad_files_do_not_break_the_batch(pipeline, sample_files, settings, tmp_path):
    empty = tmp_path / "empty.txt"
    empty.write_bytes(b"")
    exe = tmp_path / "virus.exe"
    exe.write_bytes(b"MZ")
    broken = tmp_path / "broken.pdf"
    broken.write_bytes(b"%PDF-1.4 nothing useful")
    progress = []
    results = pipeline.ingest_files([empty, sample_files["05_rag_overview.md"], exe, broken], settings,
                                    progress=lambda f, t: progress.append(f))
    assert [r.status for r in results] == ["failed", "indexed", "failed", "failed"]
    assert all(r.message for r in results if r.status == "failed")
    assert pipeline.stats()["documents"] == 1 and progress[-1] == 1.0


def test_upload_limit_per_batch(pipeline, tmp_path, settings, monkeypatch):
    monkeypatch.setattr("rag.pipeline.MAX_FILES_PER_UPLOAD", 2)
    paths = []
    for i in range(3):
        p = tmp_path / f"f{i}.txt"
        p.write_text(f"Document {i} discusses entirely different subject number {i} with enough words to index.")
        paths.append(p)
    assert [r.status for r in pipeline.ingest_files(paths, settings)] == ["indexed", "indexed", "failed"]


def test_ask_returns_grounded_answer_with_citations(loaded_pipeline, settings):
    answer = loaded_pipeline.ask("How much is the home-office stipend?", settings)
    assert answer.kind == "answer" and answer.found and "$500" in answer.text
    top = answer.sources[0]
    assert top.filename == "02_remote_work_policy.docx" and top.cited and top.number == 1
    assert 0 < top.score <= 1 and "$500" in top.text and top.page is None
    assert answer.model and answer.seconds >= 0


def test_pdf_answers_cite_the_page(loaded_pipeline, settings):
    answer = loaded_pipeline.ask("Where does the Calvin cycle take place?", settings)
    assert answer.found and "stroma" in answer.text.lower()
    cited = [s for s in answer.sources if s.cited]
    assert cited[0].filename.endswith(".pdf") and cited[0].page == 2


def test_unsupported_question_gets_honest_not_found(loaded_pipeline):
    answer = loaded_pipeline.ask("Who won the football world cup in 2018?", RAGSettings())
    assert answer.kind == "not_found" and not answer.found and answer.text == NOT_FOUND_MESSAGE


def test_empty_repository_and_invalid_queries(make_pipeline, loaded_pipeline):
    assert make_pipeline("empty").ask("What is RAG?").kind == "empty"
    assert loaded_pipeline.ask("").kind == "invalid" and loaded_pipeline.ask("??").kind == "invalid"
    assert loaded_pipeline.ask("x" * 5000).kind == "invalid"


def test_follow_up_question_uses_history(loaded_pipeline, settings):
    answer = loaded_pipeline.ask("What about the internet?", settings, history=["How much is the home-office stipend?"])
    assert "regarding" in answer.standalone_question and answer.found and "$40" in answer.text


def test_document_filter_limits_sources(loaded_pipeline, settings):
    docs = {d["filename"]: d["doc_id"] for d in loaded_pipeline.list_documents()}
    only = RAGSettings(similarity_threshold=0.0, top_k=4, doc_ids=(docs["03_renewable_energy_overview.txt"],))
    answer = loaded_pipeline.ask("What is overfitting?", only)
    assert {s.filename for s in answer.sources} <= {"03_renewable_energy_overview.txt"}


def test_delete_removes_everything_for_that_document(loaded_pipeline, settings):
    doc = next(d for d in loaded_pipeline.list_documents() if d["file_type"] == "docx")
    before = loaded_pipeline.stats()["vectors"]
    assert loaded_pipeline.delete_document(doc["doc_id"]) and not loaded_pipeline.delete_document(doc["doc_id"])
    assert loaded_pipeline.stats()["vectors"] == before - doc["chunks"]
    assert all(c["doc_id"] != doc["doc_id"] for c in loaded_pipeline.store.chunks.values())
    assert not (loaded_pipeline.upload_dir / doc["stored_as"]).exists()
    assert loaded_pipeline.ask("How much is the home-office stipend?", settings).kind == "not_found"


def test_clear_repository(loaded_pipeline):
    assert loaded_pipeline.clear_repository() == 5
    assert loaded_pipeline.stats()["documents"] == 0 and loaded_pipeline.stats()["vectors"] == 0
    assert not [p for p in loaded_pipeline.upload_dir.iterdir() if p.name != ".gitkeep"]


def test_state_survives_restart(loaded_pipeline, make_pipeline, settings):
    restarted = make_pipeline()  # same data dir
    assert restarted.stats()["documents"] == 5 and restarted.stats()["vectors"] == loaded_pipeline.stats()["vectors"]
    assert "$500" in restarted.ask("How much is the home-office stipend?", settings).text
    assert restarted.ingest_file(next(restarted.upload_dir.iterdir()), settings).status == "duplicate"


def test_reindex_applies_new_chunk_settings(loaded_pipeline, settings):
    before = loaded_pipeline.stats()["chunks"]
    results = loaded_pipeline.reindex_all(RAGSettings(chunk_size=250, chunk_overlap=40, similarity_threshold=0.12))
    assert all(r.status == "indexed" for r in results)
    stats = loaded_pipeline.stats()
    assert stats["documents"] == 5 and stats["chunks"] > before and stats["chunks"] == stats["vectors"]
    assert "$500" in loaded_pipeline.ask("How much is the home-office stipend?", settings).text


def test_invalid_settings_are_rejected(pipeline, sample_files):
    with pytest.raises(ValueError):
        pipeline.ingest_file(sample_files["05_rag_overview.md"], RAGSettings(chunk_size=50))
    with pytest.raises(ValueError):
        RAGSettings(chunk_size=300, chunk_overlap=300).validate()


def test_model_failure_degrades_to_extractive_answers(loaded_pipeline, settings):
    class Broken:
        name, is_fallback, loaded = "broken-model", False, True

        def generate(self, *a, **k):
            raise RuntimeError("CUDA error")

    loaded_pipeline.generator = Broken()
    answer = loaded_pipeline.ask("How much is the home-office stipend?", settings)
    assert answer.found and "$500" in answer.text and any("failed" in w for w in answer.warnings)


def test_generator_load_failure_is_reported(pipeline):
    class Unloadable:
        name, is_fallback, loaded = "ghost-model", False, False

        def load(self):
            raise OSError("not downloadable")

    pipeline.generator = Unloadable()
    pipeline.ensure_generator()
    assert pipeline.generator.name == "extractive-fallback" and any("ghost-model" in w for w in pipeline.warnings)


def test_safe_filename_blocks_path_tricks():
    assert safe_filename("../../etc/passwd") == "passwd"
    assert safe_filename("my report (final).pdf") == "my_report_final_.pdf"

In [ ]:
%%writefile scripts/make_samples.py
"""Create the sample document repository (Markdown, DOCX, TXT, PDF) in ./sample_docs."""
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))
from tests.pdf_utils import make_pdf  # noqa: E402

DEFAULT_OUT = ROOT / "sample_docs"

ML_MD = """# Introduction to Machine Learning

Machine learning is a branch of artificial intelligence in which computers learn patterns from data instead of following hand-written rules. A model is trained on examples, and the trained model is then used to make predictions on new, unseen data.

## Types of machine learning

**Supervised learning** uses labeled examples. Each training example has an input and the correct output, for instance an email and the label "spam" or "not spam". Common supervised tasks are classification, which predicts a category, and regression, which predicts a number such as a house price.

**Unsupervised learning** works with unlabeled data. The goal is to discover structure, such as grouping similar customers together with clustering or reducing the number of features with dimensionality reduction.

**Reinforcement learning** trains an agent that takes actions in an environment and receives rewards or penalties. Over time the agent learns a policy that maximizes the total reward. Game-playing systems and robot controllers often use this approach.

## Training, validation and test data

A dataset is usually split into three parts. The training set, commonly about 70 percent of the data, is used to fit the model. The validation set, about 15 percent, is used to tune hyperparameters and to choose between models. The test set, about 15 percent, is held back until the end to give an honest estimate of how the model will perform on new data.

## Overfitting and underfitting

Overfitting happens when a model memorizes the training data, including its noise, and therefore performs poorly on new data. Underfitting happens when a model is too simple to capture the real pattern. Techniques that reduce overfitting include collecting more data, using regularization, applying dropout in neural networks, and stopping training early when the validation error starts to rise.

## Evaluation metrics

For classification, accuracy is the share of correct predictions. Precision is the share of predicted positives that are truly positive, and recall is the share of actual positives that the model finds. The F1 score is the harmonic mean of precision and recall. For regression, common metrics are mean absolute error (MAE) and root mean squared error (RMSE).
"""

RAG_MD = """# Retrieval-Augmented Generation (RAG) Overview

Retrieval-Augmented Generation, or RAG, combines a search system with a language model. Instead of relying only on what the model memorized during training, a RAG system first retrieves relevant passages from a document collection and then asks the model to answer using those passages.

## Why use RAG

Language models can produce fluent but wrong statements, which are called hallucinations. RAG reduces hallucinations because the answer is grounded in retrieved text, and the system can show the sources it used. RAG also lets a model answer questions about private or recent documents without retraining it.

## Main components

**Chunking.** Documents are split into short passages called chunks. Neighbouring chunks usually overlap by a small amount so that a fact at a boundary is not lost.

**Embeddings.** An embedding model converts each chunk into a vector of numbers. Passages with similar meaning have vectors that are close together. The same model converts the user's question into a vector.

**Vector database.** A vector database such as FAISS stores the chunk vectors and finds the nearest ones to a query vector quickly, usually with cosine similarity.

**Generator.** A language model receives the question and the retrieved chunks and writes the answer. A good prompt tells the model to use only the supplied context and to say when the answer is missing.

## Limitations

If retrieval misses the right passage, the generator cannot answer correctly. Small models may misread long contexts, and scanned documents need optical character recognition before they can be indexed.
"""

POLICY_PARAGRAPHS = [
    ("Title", "Northwind Analytics Remote Work Policy"),
    ("Heading 1", "Purpose"),
    ("Normal", "This policy explains how employees of Northwind Analytics may work from home while keeping teams connected and company data secure."),
    ("Heading 1", "Eligibility"),
    ("Normal", "Employees become eligible for remote or hybrid work after completing the 90-day probation period. Roles that need physical access to the laboratory, such as laboratory technicians, are not eligible for remote work."),
    ("Heading 1", "Work arrangements"),
    ("Normal", "Most employees follow the hybrid model and work in the office at least 2 days per week. Tuesday and Thursday are the anchor days when every team member is expected in the office. Fully remote work needs written approval from a vice president."),
    ("Heading 1", "Core hours"),
    ("Normal", "Everyone must be available between 10:00 and 15:00 in their local time zone. Outside core hours employees may arrange their schedule with their manager."),
    ("Heading 1", "Equipment and expenses"),
    ("Normal", "The company provides a laptop to every employee. Remote employees receive a one-time home-office stipend of $500 and a monthly internet reimbursement of $40. Equipment must be returned within 14 days after an employee leaves the company."),
    ("Heading 1", "Security requirements"),
    ("Normal", "Employees must use the company VPN whenever they access internal systems. Public Wi-Fi may only be used together with the VPN. A lost or stolen device must be reported to IT security within 24 hours."),
    ("Heading 1", "Approval process"),
    ("Normal", "Requests are submitted through the HR portal. The manager decides within 5 business days and HR confirms the arrangement in writing."),
]
POLICY_TABLE = [("Expense", "Amount", "Frequency"), ("Home-office stipend", "$500", "One time"),
                ("Internet reimbursement", "$40", "Monthly"), ("Co-working space", "Not reimbursed", "Not applicable")]

ENERGY_TXT = """Renewable Energy Overview

Renewable energy comes from sources that are naturally replenished, such as sunlight, wind, flowing water and heat from the earth. Using these sources produces far fewer greenhouse gas emissions than burning coal, oil or natural gas.

Solar power. Photovoltaic panels convert sunlight directly into electricity using semiconductor cells. Commercial panels typically reach an efficiency of 18 to 22 percent. Because the sun does not shine at night and is weaker in cloudy weather, a solar plant usually has a capacity factor of only 15 to 25 percent.

Wind power. Wind turbines turn the kinetic energy of moving air into electricity. Onshore wind farms usually have a capacity factor of 30 to 40 percent, while offshore wind farms reach 40 to 50 percent because winds at sea are stronger and steadier. Offshore projects are more expensive to build and maintain.

Hydropower. Hydropower is the largest source of renewable electricity worldwide. Dams store water in a reservoir and release it through turbines. Run-of-river plants use the natural flow without a large reservoir. Pumped storage plants pump water uphill when electricity is cheap and release it when demand is high, which makes them a form of energy storage.

Geothermal and biomass. Geothermal plants use steam or hot water from underground to drive turbines and can run around the clock. Biomass plants burn organic material such as wood waste or crop residue, or convert it into biofuels.

Challenges. The main challenge of wind and solar power is intermittency: output changes with the weather and the time of day. Grid operators handle this with energy storage, such as lithium-ion batteries and pumped hydro, with long-distance transmission lines, and with demand management. Integrating large shares of variable generation also requires upgrades to the electricity grid.

Costs. The levelized cost of energy, abbreviated LCOE, is the average cost per unit of electricity over the lifetime of a power plant, including construction, fuel and maintenance. The LCOE of solar and onshore wind has fallen steadily as technology improved and manufacturing scaled up.
"""

PHOTO_PAGES = [
    """Photosynthesis and Plant Biology

Photosynthesis is the process by which plants, algae and some bacteria convert light energy into chemical energy stored in sugars. The overall reaction combines carbon dioxide and water, in the presence of light, to produce glucose and oxygen. In plants, photosynthesis takes place inside organelles called chloroplasts.

Light-dependent reactions. The light-dependent reactions occur in the thylakoid membranes of the chloroplast. Chlorophyll absorbs mostly red and blue light and reflects green light, which is why leaves look green. The absorbed energy splits water molecules, releasing oxygen as a by-product. The reactions produce the energy carriers ATP and NADPH.""",
    """The Calvin cycle

The Calvin cycle is the second stage of photosynthesis. It takes place in the stroma, the fluid-filled space around the thylakoids inside the chloroplast. The cycle does not need light directly, but it depends on the ATP and NADPH made by the light-dependent reactions.

The Calvin cycle has three stages. In carbon fixation, the enzyme RuBisCO attaches carbon dioxide to a five-carbon sugar called RuBP. In reduction, ATP and NADPH are used to convert the resulting molecules into G3P, a three-carbon sugar. In regeneration, some of the G3P is used to rebuild RuBP so the cycle can continue, while the rest is used to make glucose and other carbohydrates.""",
    """Factors that affect the rate of photosynthesis

Several factors limit how fast photosynthesis can run. Light intensity matters because more light drives the light-dependent reactions faster, until the pigments are saturated. Carbon dioxide concentration matters because RuBisCO needs carbon dioxide for carbon fixation. Temperature matters because the enzymes involved work best at moderate temperatures; above about 40 degrees Celsius many enzymes begin to denature and the rate falls sharply. Water availability matters because plants close their stomata when water is scarce, which also reduces carbon dioxide intake.

The factor that is in shortest supply at a given moment is called the limiting factor. Increasing a factor that is not limiting will not raise the rate of photosynthesis.""",
]


def main(out: Path | None = None) -> Path:
    OUT = Path(out) if out else DEFAULT_OUT
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "01_intro_to_machine_learning.md").write_text(ML_MD, encoding="utf-8")
    (OUT / "05_rag_overview.md").write_text(RAG_MD, encoding="utf-8")
    (OUT / "03_renewable_energy_overview.txt").write_text(ENERGY_TXT, encoding="utf-8")
    make_pdf(PHOTO_PAGES, OUT / "04_photosynthesis_and_plant_biology.pdf")

    import docx

    document = docx.Document()
    for style, text in POLICY_PARAGRAPHS:
        document.add_paragraph(text, style=style)
    table = document.add_table(rows=len(POLICY_TABLE), cols=3)
    table.style = "Table Grid"
    for r, row in enumerate(POLICY_TABLE):
        for col, value in enumerate(row):
            table.cell(r, col).text = value
    document.save(OUT / "02_remote_work_policy.docx")
    print("Sample documents written to", OUT)
    return OUT


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/generate_example_outputs.py
"""Run the sample questions through the pipeline and save the results to docs/example_outputs.md.

Uses the real models (MiniLM + FLAN-T5) when they can be loaded; otherwise the offline fallback
components, and the output file says which mode produced it.
"""
from __future__ import annotations

import sys
import tempfile
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

from config import RAGSettings  # noqa: E402
from rag.embeddings import HashingEmbedder  # noqa: E402
from rag.generator import ExtractiveGenerator  # noqa: E402
from rag.pipeline import DocuMindPipeline  # noqa: E402

QUESTIONS = [
    ("How much is the home-office stipend?", None),
    ("What is the difference between supervised and unsupervised learning?", None),
    ("Where does the Calvin cycle take place?", None),
    ("What is the capacity factor of offshore wind farms?", None),
    ("Why does RAG reduce hallucinations?", None),
    ("What about the internet?", "How much is the home-office stipend?"),
    ("Who won the 2018 football world cup?", None),
]


def main(real: bool = False) -> None:
    data = Path(tempfile.mkdtemp())
    if real:
        pipeline = DocuMindPipeline(data)
        pipeline.load_models()
    else:
        pipeline = DocuMindPipeline(data, embedder=HashingEmbedder(), generator=ExtractiveGenerator())
    mode = ("real models: " + pipeline.embedder.name + " + " + pipeline.generator.name) if real and not pipeline.embedder.is_fallback \
        else "OFFLINE FALLBACK (hashing embedder + extractive answers), not the real models"
    settings = RAGSettings(similarity_threshold=0.25 if real else 0.12)
    pipeline.ingest_files(sorted((ROOT / "sample_docs").glob("*")), settings)
    lines = ["# Example outputs", "", f"Generated on the sample documents. Mode: **{mode}**.", ""]
    if "FALLBACK" in mode:
        lines += ["> These outputs come from the dependency-free fallback so they could be produced without downloading models.",
                  "> With FLAN-T5 and MiniLM (the default in Colab) answers are more fluent and retrieval handles paraphrased questions.", ""]
    for question, previous in QUESTIONS:
        answer = pipeline.ask(question, settings, history=[previous] if previous else None)
        lines += [f"## Q: {question}", "", f"**Answer ({answer.kind}):** {answer.text}", ""]
        for s in answer.sources[:3]:
            page = f"page {s.page}" if s.page else "page n/a"
            flag = "cited" if s.cited else "retrieved, not cited"
            lines.append(f"- [{s.number}] `{s.filename}`, {page}, similarity {s.score:.2f} ({flag}): {s.text[:160]}...")
        lines.append("")
    out = ROOT / "docs" / "example_outputs.md"
    out.write_text("\n".join(lines), encoding="utf-8")
    print("wrote", out)


if __name__ == "__main__":
    main(real="--real" in sys.argv)

In [ ]:
%%writefile docs/example_questions.md
# Example questions for the sample documents

Load the samples with the **Load samples** button (or `python scripts/make_samples.py` and upload `sample_docs/`).

## Answerable (the answer is in the documents)

| Question | Where the answer lives |
|---|---|
| How much is the home-office stipend? | `02_remote_work_policy.docx` |
| Which days are the anchor days in the office? | `02_remote_work_policy.docx` |
| What is the difference between supervised and unsupervised learning? | `01_intro_to_machine_learning.md` |
| How is a dataset usually split for training? | `01_intro_to_machine_learning.md` |
| Where does the Calvin cycle take place? | `04_photosynthesis_and_plant_biology.pdf`, page 2 |
| What is the limiting factor in photosynthesis? | `04_photosynthesis_and_plant_biology.pdf`, page 3 |
| What is the capacity factor of offshore wind farms? | `03_renewable_energy_overview.txt` |
| What are the main challenges of wind and solar power? | `03_renewable_energy_overview.txt` |
| Why does RAG reduce hallucinations? | `05_rag_overview.md` |

## Follow-ups (ask right after the matching question)

- After "How much is the home-office stipend?" ask: "What about the internet?"
- After "Where does the Calvin cycle take place?" ask: "What are its three stages?"

## Not in the documents (DocuMind should say so)

- Who won the 2018 football world cup?
- What is the capital of Australia?
- How do I bake sourdough bread?

In [ ]:
%%writefile README.md
# DocuMind: AI-powered RAG document assistant

DocuMind lets you upload PDF, DOCX, TXT and Markdown files, ask questions in plain language, and get answers
that are generated **only** from your documents, each with the source file, page, passage and similarity score.
It runs on a free Google Colab runtime with no paid API and no API keys.

![Architecture](docs/architecture.svg)

> Screenshots: add `docs/screenshot_chat.png`, `docs/screenshot_library.png` and `docs/screenshot_settings.png` after your first run.

## 1. Problem statement

Language models write fluent text but can state things that are not true ("hallucinate"), and they know nothing
about your private files. Searching manually through many long documents is slow. DocuMind combines **search** and
**generation** so that answers are both convenient and checkable.

## 2. Objectives

1. Ingest several documents at once and keep their metadata (file, page, chunk id).
2. Retrieve the passages most relevant to a question using semantic (embedding) search.
3. Generate a short answer from those passages with a free, local model, and refuse when the answer is missing.
4. Show every answer's supporting sources so a reader can verify it.
5. Run end to end on free Colab, with tests and documentation.

## 3. Technology stack

| Part | Choice |
|---|---|
| Language / runtime | Python 3.10+, Google Colab (GPU used automatically, CPU otherwise) |
| UI | Gradio 6 with custom CSS |
| Extraction | `pypdf` (PDF), `python-docx` (DOCX), plain decoding (TXT, MD), optional Tesseract OCR |
| Cleaning / chunking | Python + `re`; custom sentence-aware overlapping chunker |
| Embeddings | Sentence-Transformers `all-MiniLM-L6-v2` (384 dimensions, normalised) |
| Vector database | FAISS (`IndexFlatIP` inside `IndexIDMap2`), JSON metadata, atomic on-disk persistence |
| LLM | `google/flan-t5-base` (switchable to small or large in Settings), PyTorch |
| Numerics | NumPy |

## 4. How RAG works (in simple terms)

* **Chunk**: a long document is cut into short passages (about 600 characters). Neighbouring chunks overlap so a
  fact that sits on a boundary still appears whole in at least one chunk.
* **Embedding**: a neural network turns each passage into a list of 384 numbers. Passages with similar meaning get
  similar numbers, even if they use different words.
* **Vector search**: the question is embedded the same way, and FAISS finds the passages whose vectors are closest.
  We use *cosine similarity* (1.0 means the same direction, 0 means unrelated). Vectors are normalised, so the inner
  product equals the cosine.
* **Threshold and reranking**: passages below the similarity threshold are dropped. The rest are re-scored with
  `0.8 x similarity + 0.2 x keyword coverage`, which rewards passages that contain the question's exact terms.
* **Grounded generation**: the model receives the best passages, numbered `[1]`, `[2]`, ... and is told to answer
  from them alone, or reply "unanswerable". Anything unanswerable becomes an honest "not found" message.
* **Citations**: after generation, DocuMind compares the answer's wording with each passage and marks the passages
  that support it. Those get filled number badges; other retrieved passages are shown but marked as not cited.

### Pipelines

```
INGESTION   Upload -> Extract -> Clean -> Chunk -> Metadata -> Embeddings -> FAISS
QUESTION    Question -> Query embedding -> Vector search -> Threshold -> Rerank -> Context -> LLM -> Answer + citations
```

## 5. Project layout

```
documind/
  app.py                  entry point (python app.py [--share])
  config.py               paths, limits, model lists, RAGSettings
  requirements.txt  pytest.ini  DocuMind_Colab.ipynb
  rag/
    loaders.py            validation, PDF/DOCX/TXT/MD extraction, OCR, cleaning
    chunking.py           sentence-aware overlapping chunks with page numbers
    embeddings.py         Sentence-Transformers wrapper, cache, hashing fallback
    vectorstore.py        FAISS + metadata registry, delete, filter, persistence
    retrieval.py          search, threshold, keyword rerank, follow-up handling
    generator.py          FLAN-T5 prompt/token budget, OOM handling, extractive fallback
    pipeline.py           ingestion and question answering orchestration
    textutils.py          shared tokenising helpers
  ui/                     components.py (theme, HTML), dashboard.py, chat.py
  data/uploads, data/vector_db   runtime storage (git-ignored content)
  sample_docs/            5 sample documents (MD, DOCX, TXT, PDF)
  scripts/                make_samples.py, generate_example_outputs.py, build_notebook.py
  tests/                  pytest suite (89 tests)
  docs/                   architecture.svg, example_questions.md, example_outputs.md
```

## 6. Installation (local)

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
python app.py            # opens http://127.0.0.1:7860 ; add --share for a public link
```
The first run downloads about 1 GB of models from Hugging Face (MiniLM and FLAN-T5 base). Optional OCR needs the
Tesseract program (`sudo apt install tesseract-ocr`).

## 7. Running in Google Colab

1. Upload `DocuMind_Colab.ipynb` to Colab (File > Upload notebook).
2. Optional: *Runtime > Change runtime type > T4 GPU*. Without a GPU everything still works, only slower.
3. Choose *Runtime > Run all*. The notebook installs dependencies, writes all source files, generates the sample
   documents, runs the tests, loads the models, and launches the app with a public `gradio.live` link.

The notebook is self-contained: it does not need this repository or any API key.

**Model trade-off.** FLAN-T5 Base (248M parameters) fits comfortably in free Colab (about 1 GB, on CPU or GPU) and
answers in a few seconds. FLAN-T5 Large gives better answers but needs the GPU runtime. FLAN-T5 Small is fastest but
makes more mistakes. Switch models under *Settings > Models*.

## 8. Using the app

1. **Upload**: choose files in the sidebar and select *Index documents* (or *Load samples*). Progress and a per-file
   result (indexed, updated, skipped duplicate, or failed with the reason) are shown.
2. **Ask**: type a question in *Chat* or use a suggested question. Follow-ups such as "What about the internet?" are
   rewritten using your previous question.
3. **Check sources**: expand each source to read the passage (query words highlighted), its page, and similarity.
4. **Manage**: delete one document, or clear the repository (tick the confirmation first). Use *Search only in
   these documents* to restrict retrieval.
5. **Tune** (Settings): chunk size and overlap (apply to newly indexed files; *Re-index* applies them to everything),
   top-K, similarity threshold, keyword reranking, answer length, OCR, and models.

Example questions and expected sources are in `docs/example_questions.md`; sample answers are in
`docs/example_outputs.md`.

## 9. Robustness and security

* Validates extension, size (15 MB), empty files, PDF/DOCX headers, password-protected and corrupted files; a bad
  file never stops the rest of a batch.
* Duplicate content is detected by SHA-256 and skipped; re-uploading a changed file with the same name replaces the
  old version, so no stale vectors remain. Deleting a document removes its vectors, metadata and stored copy.
* The vector index is written atomically and reconciled on load (orphan vectors or metadata are removed; a corrupted
  index is reset with a warning). Changing the embedding model re-indexes stored documents automatically.
* Uploaded files are copied to `data/uploads` under sanitised names; the original temp files are never modified.
* GPU memory errors are caught: embeddings split their batch, generation retries with greedy decoding and finally
  moves to CPU. If the language model cannot be loaded or crashes, answers fall back to extractive mode (copied
  passages) and a warning is shown.
* No API keys or secrets are used or stored. Documents stay on the machine running the app (the model files are
  downloaded from Hugging Face once).

## 10. Testing

```bash
python -m pytest -q          # 89 tests, about 2 seconds, no downloads
DOCUMIND_RUN_MODEL_TESTS=1 python -m pytest tests/test_generator.py   # also tests real FLAN-T5 (downloads it)
```
Tests cover extraction for all four formats, cleaning, chunk creation, embeddings (shape, normalisation, caching),
FAISS indexing, retrieval relevance, metadata preservation, empty/corrupted files, deletion, persistence, grounded
answers and honest not-found responses. The suite uses a deterministic offline embedder and extractive answerer so it
runs anywhere; the neural models are covered by the optional model test.

## 11. Limitations

* FLAN-T5 Base is small: it handles short factual questions well, but weaker at long reasoning, tables, or
  combining many passages. It reads at most about 768 tokens of context per question.
* Scanned PDFs need OCR (optional, needs Tesseract). Complex layouts (multi-column pages, tables inside PDFs) may be
  extracted in a jumbled order. DOCX and TXT have no page numbers, so citations show "Page n/a".
* Retrieval is exact (flat) search, ideal up to roughly 100,000 chunks; larger collections would need an approximate
  FAISS index.
* Citation marking is based on word overlap between the answer and passages, so it is a heuristic. A low-confidence
  warning appears when an answer barely overlaps with its sources.
* The app keeps one shared repository (not per-user) and has no login, which suits a Colab demo or single user.

## 12. Future improvements

Hybrid BM25 + vector search, a cross-encoder reranker, answer streaming, per-user repositories, approximate FAISS
indexes (IVF/HNSW) for large collections, table-aware PDF parsing, and an automatic answer-quality evaluation set.

## 5. Create the sample documents

In [ ]:
!python scripts/make_samples.py
!ls sample_docs

## 6. Run the automated tests
They use a lightweight offline embedder, so no model download is needed.

In [ ]:
!python -m pytest -q

## 7. Load the models and initialise the pipeline
Uses the GPU automatically when available.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
from rag.pipeline import DocuMindPipeline
pipeline = DocuMindPipeline()
for warning in pipeline.load_models():
    print('WARNING:', warning)
print(pipeline.status())

## 8. Launch DocuMind
Open the public `gradio.live` link below (or use the inline view). Choose **Load samples** to try it immediately.

In [ ]:
from app import build_app, launch_app
demo = build_app(pipeline)
launch_app(demo, share=True, debug=True)